# ST-SAGA: Intrinsically Interpretable Spatio-Temporal Bus Passenger Flow Forecasting with Shapley-Guided Additive Group Attribution



## 1. Libraries and Experimental Settings


### 1.1 Library Imports


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils

import numpy as np
import pandas as pd

import random
import copy
import itertools
import math
import gc
import time


### 1.2 Experimental Settings


In [ ]:
import os

BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"
DATA_DIR = f"{BASE_DIR}/data"
EXPERIMENT_NAME = "FIXED_v6_no4683"
CKD = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"

SEED = 1
N_EPOCHS = 150
PATIENCE = 20
RESUME = True

ARMS = [dict(tag="A_guided", lambda_shap=0.5, lambda_entropy=0.0)]


## 2. Data Preparation and Time-Aware Baseline


In [ ]:
def load_adjacency(adj_path):
    df = pd.read_csv(adj_path, header=None, dtype=object)
    stop_cols  = pd.to_numeric(df.iloc[0, 2:], errors="raise").astype(int).tolist()
    route_cols = pd.to_numeric(df.iloc[1, 2:], errors="raise").astype(int).tolist()
    stop_rows  = pd.to_numeric(df.iloc[3:, 0], errors="raise").astype(int).tolist()
    route_rows = pd.to_numeric(df.iloc[3:, 1], errors="raise").astype(int).tolist()
    col_pairs  = list(zip(stop_cols, route_cols))
    row_pairs  = list(zip(stop_rows, route_rows))
    if col_pairs != row_pairs:
        raise ValueError("Adjacency row/col labels are NOT in the same order.")
    labels = pd.DataFrame(col_pairs, columns=["stop_id", "route_id"])
    A = df.iloc[3:, 2:].astype(float).values
    if A.shape[0] != A.shape[1]:
        raise ValueError("Adjacency matrix is not square.")
    return labels, A

def load_flow(flow_path, labels):
    df = pd.read_csv(flow_path, header=None, dtype=object)
    stop_cols  = pd.to_numeric(df.iloc[0, 1:], errors="raise").astype(int).tolist()
    route_cols = pd.to_numeric(df.iloc[1, 1:], errors="raise").astype(int).tolist()
    flow_pairs = list(zip(stop_cols, route_cols))
    label_pairs = list(zip(labels["stop_id"].tolist(), labels["route_id"].tolist()))
    if flow_pairs != label_pairs:
        raise ValueError("Flow columns do NOT match adjacency label order.")
    X = df.iloc[3:, 1:].astype(float).values
    return X

def make_route_only_adjacency(A, labels, add_self_loops=True):
    route_ids  = labels["route_id"].astype(int).values
    same_route = (route_ids[:, None] == route_ids[None, :]).astype(np.float32)
    A_route    = A.astype(np.float32) * same_route
    if add_self_loops:
        np.fill_diagonal(A_route, 1.0)
    return A_route, same_route

def build_transfer_groups(labels):
    labels = labels.copy()
    labels["stop_id"]  = labels["stop_id"].astype(int)
    labels["route_id"] = labels["route_id"].astype(int)
    transfer_groups = []
    grouped = labels.groupby("stop_id", sort=False).groups
    for _, idxs in grouped.items():
        idxs = list(idxs)
        if labels.loc[idxs, "route_id"].nunique() >= 2:
            transfer_groups.append(idxs)
    return transfer_groups

import numpy as np, torch, torch.utils.data as utils
import pandas as pd

DAY_PERIOD = 64
TA_META    = {}

def _is_slot_table(tmpl):
    return tmpl is not None and tmpl.dim() == 2 and tmpl.shape[0] != 12

def template_for_start(slots, start, seq_len=12):
    idx = torch.as_tensor([(int(start) + t) % slots.shape[0] for t in range(seq_len)])
    return slots[idx]

def templates_for_starts(slots, starts, seq_len=12):
    return torch.stack([template_for_start(slots, int(s), seq_len) for s in starts], dim=0)

def resolve_template(tmpl, start):
    return template_for_start(tmpl, start) if _is_slot_table(tmpl) else tmpl

def PrepareDataset(flow_matrix, BATCH_SIZE=32, seq_len=12, pred_len=1,
                   train_propotion=0.7, valid_propotion=0.2):
    if isinstance(flow_matrix, pd.DataFrame):
        flow_matrix = flow_matrix.values
    flow_matrix = flow_matrix.astype(np.float32)
    flow_matrix = np.log1p(flow_matrix)
    T, N = flow_matrix.shape
    sequences, labels, starts = [], [], []
    for i in range(T - seq_len - pred_len + 1):
        sequences.append(flow_matrix[i : i + seq_len])
        labels.append(flow_matrix[i + seq_len : i + seq_len + pred_len])
        starts.append(i)
    sequences = np.asarray(sequences, dtype=np.float32)
    labels    = np.asarray(labels,    dtype=np.float32)
    starts    = np.asarray(starts,    dtype=np.int64)
    sample_size = sequences.shape[0]
    train_end   = int(np.floor(sample_size * train_propotion))
    valid_end   = int(np.floor(sample_size * (train_propotion + valid_propotion)))

    train_span = train_end + seq_len
    slot_sum = np.zeros((DAY_PERIOD, N), dtype=np.float64)
    slot_cnt = np.zeros(DAY_PERIOD, dtype=np.float64)
    for t in range(train_span):
        slot_sum[t % DAY_PERIOD] += flow_matrix[t]
        slot_cnt[t % DAY_PERIOD] += 1
    TA_META["slot_mean"] = torch.tensor(
        slot_sum / np.maximum(slot_cnt, 1)[:, None], dtype=torch.float32)
    TA_META["train_end"], TA_META["valid_end"] = train_end, valid_end
    TA_META["day_period"] = DAY_PERIOD

    def _mk(lo, hi, shuffle, drop_last):
        ds = utils.TensorDataset(torch.tensor(sequences[lo:hi]),
                                 torch.tensor(labels[lo:hi]),
                                 torch.tensor(starts[lo:hi]))
        return utils.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=drop_last)

    train_loader = _mk(0, train_end, True, True)
    valid_loader = _mk(train_end, valid_end, False, False)
    test_loader  = _mk(valid_end, sample_size, False, False)
    return train_loader, valid_loader, test_loader

def build_train_window_baseline(train_loader, seq_len=12):
    if "slot_mean" not in TA_META:
        raise RuntimeError("Run the overridden PrepareDataset first.")
    return TA_META["slot_mean"]


## 3. Graph Structure and Player Definitions


In [ ]:
PLAYER_NAMES = [
    "self_old",  "self_mid",  "self_recent",
    "route_old", "route_mid", "route_recent",
    "transfer_old", "transfer_mid", "transfer_recent",
]

WINDOWS = {
    "old":    list(range(0, 4)),
    "mid":    list(range(4, 8)),
    "recent": list(range(8, 12)),
}

def row_normalize(mat: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    row_sum = mat.sum(dim=1, keepdim=True)
    denom   = torch.where(row_sum > 0, row_sum, torch.ones_like(row_sum))
    out     = mat / (denom + eps)
    out     = torch.where(row_sum > 0, out, torch.zeros_like(out))
    return out

def build_transfer_adjacency(num_nodes: int, transfer_groups) -> torch.Tensor:
    A_transfer = torch.zeros(num_nodes, num_nodes, dtype=torch.float32)
    for grp in transfer_groups:
        grp = [int(i) for i in grp]
        for i in grp:
            for j in grp:
                if i != j:
                    A_transfer[i, j] = 1.0
    return A_transfer

def normalize_signed_masked(v: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    v_m   = v * mask
    scale = v_m.abs().sum() + eps
    return v_m / scale

def get_player_specs(model, target_node: int):
    route_nodes    = torch.where(model.route_group_mask[target_node] > 0)[0].cpu().tolist()
    transfer_nodes = torch.where(model.transfer_group_mask[target_node] > 0)[0].cpu().tolist()
    specs = {}
    for w_name, t_idx in WINDOWS.items():
        specs[f"self_{w_name}"]     = {"nodes": [target_node], "times": t_idx}
        specs[f"route_{w_name}"]    = {"nodes": route_nodes,   "times": t_idx}
        specs[f"transfer_{w_name}"] = {"nodes": transfer_nodes,"times": t_idx}
    return specs

def get_valid_player_mask(specs):
    return torch.tensor(
        [1.0 if len(specs[name]["nodes"]) > 0 else 0.0 for name in PLAYER_NAMES],
        dtype=torch.float32,
    )

def apply_player_mask(inputs, baseline_template, coalition, player_specs):
    x        = inputs.clone()
    baseline = baseline_template.to(inputs.device).unsqueeze(0).expand_as(inputs)
    inactive = set(player_specs.keys()) - set(coalition)
    for g in inactive:
        nodes = player_specs[g]["nodes"]
        times = player_specs[g]["times"]
        if len(nodes) == 0:
            continue
        for t in times:
            x[:, t, nodes] = baseline[:, t, nodes]
    return x


## 4. Route-Only GCLSTM Architecture


In [ ]:
class TransferFusion(nn.Module):
    def __init__(self, transfer_groups, in_dim, att_hidden=16,
                 overwrite=False, use_residual=True):
        super().__init__()
        self.transfer_groups = [torch.tensor(g, dtype=torch.long) for g in transfer_groups]
        self.overwrite       = overwrite
        self.use_residual    = use_residual
        self.att_net = nn.Sequential(
            nn.Linear(in_dim, att_hidden),
            nn.Tanh(),
            nn.Linear(att_hidden, 1),
        )
        if self.use_residual:
            self.beta = nn.Parameter(torch.tensor(0.0))
        self.last_alpha = []

    def forward(self, h):
        if len(self.transfer_groups) == 0:
            self.last_alpha = []
            return h
        y      = h.clone()
        self.last_alpha = []
        device = h.device
        for idxs in self.transfer_groups:
            idxs = idxs.to(device)
            if idxs.numel() < 2:
                continue
            vals   = h[:, idxs, :]
            scores = self.att_net(vals).squeeze(-1)
            alpha  = F.softmax(scores, dim=1)
            fused  = (alpha.unsqueeze(-1) * vals).sum(dim=1, keepdim=True)
            if self.overwrite and not self.use_residual:
                y[:, idxs, :] = fused.expand(-1, idxs.numel(), -1)
            elif self.use_residual:
                b = torch.sigmoid(self.beta)
                y[:, idxs, :] = (
                    (1.0 - b) * h[:, idxs, :]
                    + b * fused.expand(-1, idxs.numel(), -1)
                )
            else:
                y[:, idxs, :] = h[:, idxs, :]
            self.last_alpha.append(alpha.detach())
        return y

def make_direction_matrices(A_route_np, row_normalise=True, drop_self_loops=True, eps=1e-9):
    A = torch.as_tensor(A_route_np, dtype=torch.float32).clone()
    if drop_self_loops: A.fill_diagonal_(0.0)
    A_up, A_dn = A.t().contiguous(), A.contiguous()
    if row_normalise:
        def rn(M):
            d = M.sum(dim=1, keepdim=True)
            return torch.where(d > 0, M / (d + eps), torch.zeros_like(M))
        A_up, A_dn = rn(A_up), rn(A_dn)
    return A_up, A_dn


class IntrinsicShapleyRouteOnlyGCLSTM(nn.Module):
    def __init__(self, K, A_route, route_allowed_mask, num_nodes, embed_dim,
                 transfer_groups, pred_len=1, transfer_att_hidden=16,
                 use_down_channel=False):
        super().__init__()
        if pred_len != 1:
            raise ValueError("This implementation is written for pred_len = 1.")
        self.N, self.D, self.K, self.pred_len = num_nodes, embed_dim, K, pred_len
        self.use_down_channel = use_down_channel

        self.register_buffer("A_route_base",       torch.tensor(A_route,            dtype=torch.float32))
        self.register_buffer("route_allowed_mask", torch.tensor(route_allowed_mask, dtype=torch.float32))

        route_group_mask = torch.tensor(route_allowed_mask, dtype=torch.float32)
        route_group_mask.fill_diagonal_(0.0)
        transfer_group_mask = build_transfer_adjacency(num_nodes, transfer_groups)
        self.register_buffer("route_group_mask",    route_group_mask)
        self.register_buffer("transfer_group_mask", transfer_group_mask)
        self.register_buffer("route_group_norm",    row_normalize(route_group_mask))
        self.register_buffer("transfer_group_norm", row_normalize(transfer_group_mask))
        self.register_buffer("route_valid_mask",    (route_group_mask.sum(dim=1) > 0).float())
        self.register_buffer("transfer_valid_mask", (transfer_group_mask.sum(dim=1) > 0).float())
        self.transfer_target_nodes = torch.where(self.transfer_valid_mask > 0)[0].cpu().tolist()

        A_up, A_dn = make_direction_matrices(A_route)
        self.register_buffer("A_up", A_up)
        self.register_buffer("A_dn", A_dn)

        self.x_proj = nn.Linear(1, self.D)
        self.transfer_fusion = TransferFusion(
            transfer_groups=transfer_groups, in_dim=self.D,
            att_hidden=transfer_att_hidden, overwrite=False, use_residual=True)
        self.lin_up = nn.Linear(self.D, self.D)
        self.lin_dn = nn.Linear(self.D, self.D) if use_down_channel else None
        self.post_mp_beta = nn.Parameter(torch.tensor(0.0))

        n_chan = 2 if use_down_channel else 1
        gate_in_dim = self.D * (n_chan + 2)
        self.f_gate = nn.Linear(gate_in_dim, self.D)
        self.i_gate = nn.Linear(gate_in_dim, self.D)
        self.o_gate = nn.Linear(gate_in_dim, self.D)
        self.g_gate = nn.Linear(gate_in_dim, self.D)

        self.branch_heads = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(self.D, self.D), nn.ReLU(),
                                nn.Linear(self.D, self.pred_len))
            for name in PLAYER_NAMES})
        self.output_bias = nn.Parameter(torch.zeros(self.pred_len, self.N))
        self.last_H_seq = None
        self.last_branch_outs = None

    def _mp(self, x, A_norm):
        return torch.einsum("ij,bjd->bid", A_norm, x)

    def _aggregate_group(self, H_win, group_norm):
        return torch.einsum("ij,bjd->bid", group_norm, H_win)

    def _channel(self, x_emb, A_dir, lin, beta_post):
        h       = lin(self._mp(x_emb, A_dir)) + x_emb
        h_fused = self.transfer_fusion(h)
        h_post  = self._mp(h_fused, A_dir)
        return (1.0 - beta_post) * h_fused + beta_post * h_post

    def forward(self, inputs, return_branch_outs=False):
        self.last_H_seq = None; self.last_branch_outs = None
        B, T, N = inputs.shape
        if N != self.N:
            raise ValueError(f"Input has N={N}, but model expects N={self.N}")
        if T != 12:
            raise ValueError("This implementation assumes seq_len = 12.")
        device = inputs.device
        A_up, A_dn = self.A_up.to(device), self.A_dn.to(device)

        H = torch.zeros(B, self.N, self.D, device=device)
        C = torch.zeros(B, self.N, self.D, device=device)
        H_seq = []
        beta_post = torch.sigmoid(self.post_mp_beta)

        for t in range(T):
            x_emb = self.x_proj(inputs[:, t, :].unsqueeze(-1))
            chans = [self._channel(x_emb, A_up, self.lin_up, beta_post)]
            if self.use_down_channel:
                chans.append(self._channel(x_emb, A_dn, self.lin_dn, beta_post))
            combined = torch.cat([x_emb] + chans + [H], dim=-1)
            f = torch.sigmoid(self.f_gate(combined)); i = torch.sigmoid(self.i_gate(combined))
            o = torch.sigmoid(self.o_gate(combined)); g = torch.tanh(self.g_gate(combined))
            C = f * C + i * g
            H = o * torch.tanh(C)
            H_seq.append(H.unsqueeze(1))
        H_seq = torch.cat(H_seq, dim=1)

        H_old    = H_seq[:, WINDOWS["old"],    :, :].mean(dim=1)
        H_mid    = H_seq[:, WINDOWS["mid"],    :, :].mean(dim=1)
        H_recent = H_seq[:, WINDOWS["recent"], :, :].mean(dim=1)

        group_embs = {
            "self_old": H_old, "self_mid": H_mid, "self_recent": H_recent,
            "route_old":       self._aggregate_group(H_old,    self.route_group_norm.to(device)),
            "route_mid":       self._aggregate_group(H_mid,    self.route_group_norm.to(device)),
            "route_recent":    self._aggregate_group(H_recent, self.route_group_norm.to(device)),
            "transfer_old":    self._aggregate_group(H_old,    self.transfer_group_norm.to(device)),
            "transfer_mid":    self._aggregate_group(H_mid,    self.transfer_group_norm.to(device)),
            "transfer_recent": self._aggregate_group(H_recent, self.transfer_group_norm.to(device)),
        }

        branch_list = []
        for name in PLAYER_NAMES:
            c = self.branch_heads[name](group_embs[name]).permute(0, 2, 1)
            if name.startswith("route_"):
                c = c * self.route_valid_mask.to(device).view(1, 1, self.N)
            elif name.startswith("transfer_"):
                c = c * self.transfer_valid_mask.to(device).view(1, 1, self.N)
            branch_list.append(c)

        branch_outs = torch.stack(branch_list, dim=1)
        y = branch_outs.sum(dim=1) + self.output_bias.unsqueeze(0)

        if return_branch_outs:
            self.last_H_seq = H_seq.detach()
            self.last_branch_outs = branch_outs.detach()
            return y, branch_outs
        del H_seq
        return y

    def compute_intrinsic_regularizer(self, branch_outs, lambda_l1=1e-3,
                                      lambda_entropy=1e-3, eps=1e-8):
        imp = branch_outs.abs()
        tot = imp.sum(dim=1, keepdim=True)
        p   = imp / (tot + eps)
        ent = -(p * torch.log(p + 1e-12)).sum(dim=1)
        ent = torch.where(tot.squeeze(1) > eps, ent, torch.zeros_like(ent))
        return lambda_l1 * branch_outs.abs().mean() + lambda_entropy * ent.mean()


## 5. Reference Contributions and Shapley Alignment


In [ ]:
def reference_branch_outs(model, inputs, slots_or_tmpl, starts, return_pred=True):
    device = inputs.device
    if _is_slot_table(slots_or_tmpl):
        tb = templates_for_starts(slots_or_tmpl, starts).to(device)
    else:
        tb = slots_or_tmpl.unsqueeze(0).expand(inputs.size(0), -1, -1).to(device)
    y,  branch_x    = model(inputs, return_branch_outs=True)
    v0, branch_base = model(tb,     return_branch_outs=True)
    c_tilde = branch_x - branch_base
    model.last_H_seq = None; model.last_branch_outs = None
    return (c_tilde, y, v0) if return_pred else c_tilde

def reference_branch_outs_target(model, x_one, tmpl, target_node):
    specs = get_player_specs(model, target_node)
    x_ref = apply_player_mask(x_one, tmpl, set(), specs)
    y,    branch_x   = model(x_one, return_branch_outs=True)
    yref, branch_ref = model(x_ref, return_branch_outs=True)
    return (branch_x - branch_ref)[0, :, 0, target_node], y, yref

SHAP_ALIGN_MODE = "rel_l2"

SHAP_DEN_FLOOR = 1e-4

SHAP_LOSS_CAP  = 25.0

def shapley_alignment_loss_v2(model, c_vec, phi_dict, target_node, mode=None):
    mode = mode or SHAP_ALIGN_MODE
    dev  = c_vec.device
    vm   = get_valid_player_mask(get_player_specs(model, target_node)).to(dev)
    phi  = torch.tensor([phi_dict[n] for n in PLAYER_NAMES], dtype=torch.float32, device=dev)
    if float((phi * vm).abs().sum()) < 1e-8:
        return torch.zeros((), device=dev)
    if mode == "cosine":
        return 1.0 - F.cosine_similarity(
            normalize_signed_masked(c_vec, vm).unsqueeze(0),
            normalize_signed_masked(phi,   vm).unsqueeze(0)).squeeze()
    m = vm > 0

    if mode == "rel_l2":
        den = (phi[m] ** 2).sum().detach().clamp(min=SHAP_DEN_FLOOR)
        return (((c_vec[m] - phi[m]) ** 2).sum() / den).clamp(max=SHAP_LOSS_CAP)
    if mode == "rel_l1":
        den = phi[m].abs().sum().detach().clamp(min=SHAP_DEN_FLOOR ** 0.5)
        return ((c_vec[m] - phi[m]).abs().sum() / den).clamp(max=SHAP_LOSS_CAP)
    raise ValueError(f"unknown SHAP_ALIGN_MODE: {mode}")

SHAP_COAL_BATCH = 128

@torch.no_grad()
def estimate_group_shapley(model, inputs, target_node, baseline_template,
                           exact=False, num_permutations=50, coal_batch=None):
    if inputs.size(0) != 1:
        raise ValueError("estimate_group_shapley expects shape (1, T, N)")
    if inputs.size(1) != 12:
        raise ValueError("This implementation assumes seq_len = 12.")
    coal_batch    = coal_batch or SHAP_COAL_BATCH
    specs         = get_player_specs(model, target_node)
    valid_mask    = get_valid_player_mask(specs)
    valid_players = [n for i, n in enumerate(PLAYER_NAMES) if valid_mask[i] > 0]
    phi           = {g: 0.0 for g in PLAYER_NAMES}
    was_training  = model.training
    model.eval()
    cache = {}

    def value_many(coalitions):
        todo = [c for c in coalitions if c not in cache]
        for s in range(0, len(todo), coal_batch):
            chunk = todo[s:s + coal_batch]
            xb = torch.cat([apply_player_mask(inputs, baseline_template, set(c), specs)
                            for c in chunk], dim=0)
            yb = model(xb)
            for k, c in enumerate(chunk):
                cache[c] = float(yb[k, 0, target_node].item())
            del xb, yb
        model.last_H_seq = None; model.last_branch_outs = None

    if exact:
        M = len(valid_players)
        value_many([frozenset(c) for r in range(M + 1)
                    for c in itertools.combinations(valid_players, r)])
        for p in valid_players:
            others = [q for q in valid_players if q != p]
            for r in range(len(others) + 1):
                for S in itertools.combinations(others, r):
                    S = frozenset(S)
                    w = (math.factorial(len(S)) * math.factorial(M - len(S) - 1)
                         / math.factorial(M))
                    phi[p] += w * (cache[S | {p}] - cache[S])
    else:
        for _ in range(num_permutations):
            perm = valid_players[:]; random.shuffle(perm)
            prefixes, S = [frozenset()], set()
            for p in perm:
                S = S | {p}; prefixes.append(frozenset(S))
            value_many(prefixes)
            prev = cache[prefixes[0]]
            for k, p in enumerate(perm, start=1):
                cur = cache[prefixes[k]]
                phi[p] += (cur - prev); prev = cur
        for p in valid_players:
            phi[p] /= float(num_permutations)

    if was_training: model.train()
    del cache
    model.last_H_seq = None; model.last_branch_outs = None
    gc.collect()
    return phi


## 6. Training Procedure


In [ ]:
def TrainModel(
    model, train_loader, valid_loader, baseline_template,
    learning_rate=1e-3, num_epochs=150, patience=20, min_delta=1e-6,
    lambda_l1=1e-3, lambda_entropy=1e-3, lambda_shap=1e-3,
    shap_warmup_epochs=5, shap_every_batches=50,
    shap_exact=True, shap_num_permutations=50,
    shap_updates_per_epoch=20, **_ignored,
):
    if _ignored:
        print(f"  (ignoring obsolete args: {sorted(_ignored)})", flush=True)
    device            = "cuda" if torch.cuda.is_available() else "cpu"
    model             = model.to(device)
    baseline_template = baseline_template.to(device)

    _n_batches = len(train_loader)
    _every = (max(1, _n_batches // shap_updates_per_epoch)
              if shap_updates_per_epoch else shap_every_batches)
    print(f"  align_mode={SHAP_ALIGN_MODE} | lambda_shap={lambda_shap} | "
          f"lambda_l1={lambda_l1}", flush=True)
    print(f"  {_n_batches} batches/epoch -> guiding every {_every} "
          f"(~{len(range(0,_n_batches,_every))} updates/epoch; was "
          f"{len(range(0,_n_batches,shap_every_batches))})", flush=True)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_valid, bad_epochs = float("inf"), 0
    best_state = copy.deepcopy(model.state_dict())
    history    = {"train_mse": [], "valid_mse": []}

    for epoch in range(num_epochs):
        model.train()
        tr_pred, tr_reg, tr_shap, tr_gap = [], [], [], []
        tr_shap_active = []

        for batch_idx, (inputs, targets, starts) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)

            c_tilde, outputs, v0 = reference_branch_outs(
                model, inputs, baseline_template, starts)
            pred_loss = criterion(outputs, targets)
            reg_loss  = model.compute_intrinsic_regularizer(
                c_tilde, lambda_l1=lambda_l1, lambda_entropy=lambda_entropy)

            with torch.no_grad():
                gap = (c_tilde.sum(dim=1) - (outputs - v0)).abs().mean()

            shap_loss = torch.tensor(0.0, device=device)
            use_shap = (lambda_shap > 0 and epoch >= shap_warmup_epochs
                        and batch_idx % _every == 0)
            if use_shap:
                sample_idx = random.randrange(inputs.size(0))
                if len(model.transfer_target_nodes) > 0 and random.random() < 0.5:
                    target_node = random.choice(model.transfer_target_nodes)
                else:
                    target_node = random.randrange(model.N)
                x_one  = inputs[sample_idx : sample_idx + 1]
                tmpl_s = resolve_template(baseline_template, starts[sample_idx]).to(device)
                phi_dict = estimate_group_shapley(
                    model=model, inputs=x_one.detach(), target_node=target_node,
                    baseline_template=tmpl_s, exact=shap_exact,
                    num_permutations=shap_num_permutations)
                c_vec, _, _ = reference_branch_outs_target(
                    model, x_one, tmpl_s, target_node)
                shap_loss = shapley_alignment_loss_v2(
                    model=model, c_vec=c_vec, phi_dict=phi_dict,
                    target_node=target_node)
                del phi_dict, x_one, c_vec

            loss = pred_loss + reg_loss + lambda_shap * shap_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_pred.append(pred_loss.item()); tr_reg.append(reg_loss.item())
            tr_shap.append(float(shap_loss.item())); tr_gap.append(gap.item())
            if use_shap: tr_shap_active.append(float(shap_loss.item()))
            del c_tilde, outputs, v0, pred_loss, reg_loss, shap_loss, loss, gap
            if batch_idx % 10 == 0: gc.collect()

        model.eval()
        valid_losses = []
        with torch.no_grad():
            for val_inputs, val_targets, _ in valid_loader:
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                valid_losses.append(criterion(model(val_inputs), val_targets).item())
                del val_inputs, val_targets

        avg_train, avg_valid = float(np.mean(tr_pred)), float(np.mean(valid_losses))
        history["train_mse"].append(avg_train); history["valid_mse"].append(avg_valid)
        _reg  = float(np.mean(tr_reg))
        _shpA = float(np.mean(tr_shap_active)) if tr_shap_active else float("nan")
        _shpE = float(np.mean(tr_shap))
        print(f"Epoch {epoch+1:03d} | Train MSE: {avg_train:.6f} | Valid MSE: {avg_valid:.6f} | "
              f"EffGap: {np.mean(tr_gap):.2e}", flush=True)
        print(f"          L_pred {avg_train:9.6f} 100.0%  |  "
              f"L_reg {_reg:9.6f} {100*_reg/max(avg_train,1e-12):5.2f}%  |  "
              f"lam*L_shp {lambda_shap*_shpA:9.6f} "
              f"{100*lambda_shap*_shpA/max(avg_train,1e-12):5.2f}% on {len(tr_shap_active)} "
              f"guided batches ({100*lambda_shap*_shpE/max(avg_train,1e-12):5.2f}% epoch avg)",
              flush=True)

        if (lambda_shap <= 0) or (epoch >= shap_warmup_epochs):
            if (best_valid - avg_valid) > min_delta:
                best_valid, best_state, bad_epochs = avg_valid, copy.deepcopy(model.state_dict()), 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"Early stopping at epoch {epoch + 1}"); break
        gc.collect()

    model.load_state_dict(best_state)
    return model, history


## 7. Prediction Metrics and Checkpoints


In [ ]:
def TestModel(model, test_loader):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = model.to(device); model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs, labels = batch[0].to(device), batch[1]
            preds.append(model(inputs).cpu().numpy()); trues.append(labels.numpy())
            del inputs
    preds, trues = np.concatenate(preds, 0), np.concatenate(trues, 0)
    preds_orig, trues_orig = np.expm1(preds), np.expm1(trues)
    mae  = float(np.mean(np.abs(preds_orig - trues_orig)))
    rmse = float(np.sqrt(np.mean((preds_orig - trues_orig) ** 2)))
    return mae, rmse

def save_checkpoint(model, baseline_template, history, path, test_loader=None, labels=None):
    transfer_groups = [g.cpu().tolist() for g in model.transfer_fusion.transfer_groups]
    att_hidden      = model.transfer_fusion.att_net[0].out_features
    ckpt = {
        "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
        "config": {
            "K":                   model.K,
            "num_nodes":           model.N,
            "embed_dim":           model.D,
            "pred_len":            model.pred_len,
            "transfer_att_hidden": att_hidden,
            "use_down_channel":    bool(getattr(model, "use_down_channel", True)),
        },
        "A_route":            model.A_route_base.cpu().numpy(),
        "route_allowed_mask": model.route_allowed_mask.cpu().numpy(),
        "transfer_groups":    transfer_groups,
        "baseline_template":  baseline_template.detach().cpu(),
        "history":            history,
    }
    if labels is not None:
        ckpt["labels"] = {
            "stop_ids":  [int(s) for s in labels["stop_id"]],
            "route_ids": [int(r) for r in labels["route_id"]],
        }
    if test_loader is not None:
        ckpt["test_data"]  = test_loader.dataset.tensors[0].cpu()
        ckpt["test_label"] = test_loader.dataset.tensors[1].cpu()
    import os as _os
    _parent = _os.path.dirname(path)
    if _parent:
        _os.makedirs(_parent, exist_ok=True)
    torch.save(ckpt, path)
    print(f"Checkpoint saved -> {path}", flush=True)

def _build_model_from_ckpt(ckpt, device=None):
    cfg  = ckpt["config"]
    model = IntrinsicShapleyRouteOnlyGCLSTM(
        K=cfg["K"],
        A_route=ckpt["A_route"],
        route_allowed_mask=ckpt["route_allowed_mask"],
        num_nodes=cfg["num_nodes"],
        embed_dim=cfg["embed_dim"],
        transfer_groups=ckpt["transfer_groups"],
        pred_len=cfg["pred_len"],
        transfer_att_hidden=cfg["transfer_att_hidden"],
        use_down_channel=cfg.get("use_down_channel", True),
    )
    model.load_state_dict(ckpt["state_dict"])
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    baseline_template = ckpt["baseline_template"].to(device)
    return model, baseline_template, device

def load_checkpoint(path, device=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model, baseline_template, device = _build_model_from_ckpt(ckpt, device)
    print(f"Checkpoint loaded <- {path} (device={device})", flush=True)
    return model, baseline_template, ckpt.get("history")


## 8. Training and Test Evaluation


In [ ]:
def _ab_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def _ab_build(seed):
    _ab_seed(seed)
    labels, A = load_adjacency(f"{DATA_DIR}/ames_adjacency_directed_no4683.csv")
    X         = load_flow(f"{DATA_DIR}/ames_final_main_matrix_no4683.csv", labels)
    A_route, same_route = make_route_only_adjacency(A, labels, add_self_loops=True)
    tr, va, te = PrepareDataset(X, BATCH_SIZE=16, seq_len=12, pred_len=1,
                                train_propotion=0.7, valid_propotion=0.2)
    slots = build_train_window_baseline(tr, seq_len=12)
    model = IntrinsicShapleyRouteOnlyGCLSTM(
        K=1, A_route=A_route, route_allowed_mask=same_route,
        num_nodes=X.shape[1], embed_dim=32,
        transfer_groups=build_transfer_groups(labels),
        pred_len=1, transfer_att_hidden=32, use_down_channel=False)
    return model, slots, tr, va, te, labels

for arm in ARMS:
    tag = arm["tag"]

    print(f"\n{'='*66}\n  ARM {tag}  seed {SEED}  "
          f"lambda_shap={arm['lambda_shap']}\n{'='*66}", flush=True)

    model, slots, tr, va, te, labels = _ab_build(SEED)

    ckpt = f"{CKD}/ablation_{tag}_seed{SEED}.pt"

    if RESUME and os.path.exists(ckpt):
        print(f"  RESUME: loading {ckpt}", flush=True)
        best, slots_ck, hist = load_checkpoint(ckpt)
        if slots_ck is not None:
            slots = slots_ck
        _raw = torch.load(ckpt, map_location="cpu", weights_only=False)
        _a, _b = _raw["test_data"].float(), te.dataset.tensors[0].float()
        assert _a.shape == _b.shape and torch.allclose(_a, _b), (
            f"RESUME ABORTED: {ckpt} was trained on different data. "
            f"Delete it or set RESUME=False.")
        print(f"  data check OK {tuple(_a.shape)}", flush=True)
        del _raw, _a, _b
        mins = 0.0
    else:
        t0 = time.time()
        best, hist = TrainModel(
            model=model, train_loader=tr, valid_loader=va, baseline_template=slots,
            learning_rate=0.00302402517740569, num_epochs=N_EPOCHS, patience=PATIENCE,
            lambda_l1=0.0, lambda_entropy=arm["lambda_entropy"],
            lambda_shap=arm["lambda_shap"],
            shap_warmup_epochs=15, shap_every_batches=200,
            shap_updates_per_epoch=10, shap_exact=True, shap_num_permutations=50)
        mins = (time.time() - t0) / 60
        save_checkpoint(best, slots, hist, ckpt, test_loader=te, labels=labels)

    mae, rmse = TestModel(best, te)
    print(f"Test MAE: {mae:.4f} | Test RMSE: {rmse:.4f}")


## 9. Attribution Groups and Node Labels


In [ ]:
IDX_SELF     = [0, 1, 2]

IDX_ROUTE    = [3, 4, 5]

IDX_TRANSFER = [6, 7, 8]

IDX_OLD      = [0, 3, 6]

IDX_MID      = [1, 4, 7]

IDX_RECENT   = [2, 5, 8]

PLAYER_COLORS = {
    "self_old":       "#2c7bb6",
    "self_mid":       "#abd9e9",
    "self_recent":    "#74add1",
    "route_old":      "#d7191c",
    "route_mid":      "#fdae61",
    "route_recent":   "#f46d43",
    "transfer_old":   "#1a9641",
    "transfer_mid":   "#a6d96a",
    "transfer_recent":"#66bd63",
}

SPATIAL_GROUP_PLAYERS = [
    [PLAYER_NAMES[i] for i in IDX_SELF],
    [PLAYER_NAMES[i] for i in IDX_ROUTE],
    [PLAYER_NAMES[i] for i in IDX_TRANSFER],
]

TEMPORAL_GROUP_PLAYERS = [
    [PLAYER_NAMES[i] for i in IDX_OLD],
    [PLAYER_NAMES[i] for i in IDX_MID],
    [PLAYER_NAMES[i] for i in IDX_RECENT],
]

def _node_caption(node_idx, node_meta=None):
    base = f"Node {node_idx}"
    if node_meta is None:
        return base
    try:
        sid = node_meta["stop_ids"][node_idx]
        rid = node_meta["route_ids"][node_idx]
        return f"{base} (stop {sid}, route {rid})"
    except Exception:
        return base


## 10. Local Shapley Attribution


### 10.1 Local Shapley Decomposition


In [ ]:
import os as _os
import math as _math
import numpy as _np
import torch as _torch
import matplotlib.pyplot as _plt
import matplotlib.patches as _mpatches

_need = ["reference_branch_outs_target", "estimate_group_shapley",
         "get_player_specs", "get_valid_player_mask", "resolve_template",
         "_node_caption", "PLAYER_NAMES", "PLAYER_COLORS",
         "IDX_SELF", "IDX_ROUTE", "IDX_TRANSFER",
         "IDX_OLD", "IDX_MID", "IDX_RECENT"]
_miss = [f for f in _need if f not in globals()]
if _miss:
    raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_miss))


def _bars(ax, vals, labels, colors, title, xlabel, valid=None, fs=9):
    y    = _np.arange(len(labels))[::-1]
    ok   = [1] * len(vals) if valid is None else valid
    bars = ax.barh(y, vals, height=0.62, edgecolor="white", linewidth=0.8,
                   color=[c if ok[i] else "#cfcfcf" for i, c in enumerate(colors)])
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=fs)
    ax.set_xlabel(xlabel, fontsize=9.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.grid(axis="x", linestyle="--", alpha=0.35); ax.set_axisbelow(True)
    span = max(1e-3, max((abs(v) for v in vals), default=0.0))
    for b, v, k in zip(bars, vals, ok):
        ax.text((v if v >= 0 else 0.0) + 0.035 * span,
                b.get_y() + b.get_height() / 2,
                "N/A" if not k else f"{v:+.4f}",
                va="center", ha="left", fontsize=8,
                color="#9a9a9a" if not k else "#222222")
    pad = 0.30 * span
    ax.set_xlim(min(0.0, min(vals)) - (pad if min(vals) < 0 else 0.04 * span),
                max(0.0, max(vals)) + pad)


@_torch.no_grad()
def explain_target_node_phi_min(model, inputs, target_node, slots, start,
                                exact=True, num_permutations=50,
                                save_path="explanation_min.png", node_meta=None,
                                show_ctilde=True, residual_tol=1e-3,
                                verbose=True, close=False):
    if inputs.dim() != 3 or inputs.size(0) != 1:
        raise ValueError(f"inputs must be (1, 12, N); got {tuple(inputs.shape)}")
    if not 0 <= target_node < model.N:
        raise IndexError(f"target_node must be in [0, {model.N - 1}]")

    dev, was_training = next(model.parameters()).device, model.training
    model.eval()
    inputs = inputs.to(dev)
    tmpl   = resolve_template(slots, int(start)).to(dev)
    if tmpl.dim() != 2 or tmpl.size(0) != inputs.size(1):
        raise ValueError(f"template must be ({inputs.size(1)}, N); got {tuple(tmpl.shape)}")

    c_vec, y, yref = reference_branch_outs_target(model, inputs, tmpl, target_node)
    c    = c_vec.detach().cpu().numpy().astype(_np.float64)
    f    = float(y[0, 0, target_node].item())
    phi0 = float(yref[0, 0, target_node].item())

    phi_d = estimate_group_shapley(model=model, inputs=inputs, target_node=target_node,
                                   baseline_template=tmpl, exact=exact,
                                   num_permutations=num_permutations)
    phi   = _np.array([float(phi_d.get(n, 0.0)) for n in PLAYER_NAMES], dtype=_np.float64)
    vmask = get_valid_player_mask(get_player_specs(model, target_node)).cpu().numpy().astype(bool)

    residual = abs(phi0 + phi.sum() - f)
    c_resid  = abs(phi0 + c.sum() - f)
    if residual > residual_tol:
        raise RuntimeError(f"LOCAL ACCURACY VIOLATED: |phi0 + sum(phi) - f| = "
                           f"{residual:.3e} > {residual_tol:.0e}. phi0={phi0:.6f} "
                           f"sum(phi)={phi.sum():.6f} f={f:.6f}. Check slots/start.")

    spatial  = _np.array([phi[i].sum() for i in (IDX_SELF, IDX_ROUTE, IDX_TRANSFER)])
    temporal = _np.array([phi[i].sum() for i in (IDX_OLD, IDX_MID, IDX_RECENT)])
    sp_ok    = [1, int(vmask[IDX_ROUTE[0]]), int(vmask[IDX_TRANSFER[0]])]
    spatial  = _np.where(_np.array(sp_ok, dtype=bool), spatial, 0.0)

    b_pred, b_base = float(_np.expm1(f)), float(_np.expm1(phi0))
    mult = float(_math.exp(phi.sum()))
    neg  = b_pred < 0 or b_base < 0
    if neg:
        print(f"  NOTE node {target_node}: boardings {b_base:.2f} -> {b_pred:.2f}. "
              f"log1p is unbounded below, so the model can go negative at a quiet "
              f"stop. The decomposition is still exact, but pick a busier stop.")
    if verbose:
        print(f"  node {target_node} | start {int(start)} | "
              f"{'EXACT' if exact else f'{num_permutations} perms'} | "
              f"valid players {int(vmask.sum())}")
        print(f"    phi0 {phi0:.6f} + sum(phi) {phi.sum():+.6f} = {phi0 + phi.sum():.6f}"
              f"  vs f {f:.6f}   residual {residual:.2e}  (c~ {c_resid:.2e})")
        print(f"    boardings {b_base:.2f} -> {b_pred:.2f} "
              f"(raw {100 * (b_pred / b_base - 1):+.1f}%)  |  "
              f"exp(sum phi) = x{mult:.3f} on (1 + boardings)")

    pcol  = [PLAYER_COLORS[n] for n in PLAYER_NAMES]
    pval  = [int(v) for v in vmask]
    panes = ([("c", list(c), PLAYER_NAMES, pcol, "1 · Intrinsic  c̃ₚ",
               "log-space deviation vs empty coalition", pval, 9)] if show_ctilde else []) + [
             ("p", list(phi), PLAYER_NAMES, pcol,
              "Exact Shapley  φₚ" if exact else f"Shapley φₚ ({num_permutations} perms)",
              "log-space φ value", pval, 9),
             ("s", list(spatial), ["Self", "Route", "Transfer"],
              ["#2c7bb6", "#d7191c", "#1a9641"], "Spatial — WHERE",
              "Σ of the nine φ, by location  (log-space)", sp_ok, 10),
             ("t", list(temporal), ["Old\n(t−11…t−8)", "Mid\n(t−7…t−4)", "Recent\n(t−3…t)"],
              ["#756bb1", "#9ebcda", "#2b8cbe"], "Temporal — WHEN",
              "Σ of the nine φ, by recency  (log-space)", None, 10)]
    ncol = 2 if len(panes) == 4 else 3
    nrow = 2 if len(panes) == 4 else 1
    fig, axes = _plt.subplots(nrow, ncol, figsize=(6.4 * ncol, 4.4 * nrow))
    axes = _np.atleast_1d(axes).ravel()
    for ax, (_, v, lab, col, ttl, xl, ok, fs) in zip(axes, panes):
        _bars(ax, v, lab, col, ttl, xl, ok, fs)
    box = dict(boxstyle="round,pad=0.30", facecolor="#f2f2f2",
               edgecolor="#cccccc", linewidth=0.7)
    for ax, (kind, v, *_r) in zip(axes, panes):
        tot = c.sum() if kind == "c" else (phi.sum() if kind == "p" else sum(v))
        sym = "Σ c̃" if kind == "c" else "Σ φ"
        ax.text(0.98, 0.04, f"{sym} = {tot:+.4f}", transform=ax.transAxes,
                ha="right", va="bottom", fontsize=9.5, fontweight="bold", bbox=box)

    fig.suptitle(
        f"{_node_caption(target_node, node_meta)}   —   empty-coalition prediction "
        f"{b_base:.2f} → model prediction {b_pred:.2f} boardings"
        + ("   [model output is negative here — log1p is unbounded below]" if neg else "")
        + f"\nlog space:  φ₀ = {phi0:.3f}   Σφ = {phi.sum():+.3f}   "
          f"φ₀ + Σφ = {phi0 + phi.sum():.3f} = f   (residual {residual:.1e})"
          f"      exp(Σφ) = ×{mult:.3f} on (1 + boardings)",
        fontsize=12.5, fontweight="bold", y=1.005)
    fig.legend(handles=[_mpatches.Patch(color="#2c7bb6", label="Self"),
                        _mpatches.Patch(color="#d7191c", label="Route peers"),
                        _mpatches.Patch(color="#1a9641", label="Transfer peers"),
                        _mpatches.Patch(color="#cfcfcf", label="player absent (N/A)")],
               loc="lower center", ncol=4, fontsize=9, frameon=False,
               bbox_to_anchor=(0.5, -0.05))
    fig.text(0.5, -0.085,
             "The WHERE and WHEN panels partition the SAME nine φ, so all totals agree. They are aggregated "
             "nine-player Shapley contributions, NOT the Shapley values of a separate 3-player game.\n"
             "φ values ADD in log space only. φ₀ is the model's prediction with this stop's nine players "
             "masked to their slot mean; every other stop is unchanged.",
             ha="center", fontsize=8.2, color="#777777")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    d = _os.path.dirname(save_path)
    if d:
        _os.makedirs(d, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    if close:
        _plt.close(fig)
    else:
        try:
            _plt.show()
        except Exception:
            pass
    if verbose:
        print(f"    figure saved -> {save_path}", flush=True)

    model.last_H_seq = model.last_branch_outs = None
    if was_training:
        model.train()
    return fig, dict(
        target_node=int(target_node), start=int(start), exact=bool(exact),
        c_tilde={n: float(c[i]) for i, n in enumerate(PLAYER_NAMES)},
        phi={n: float(phi[i]) for i, n in enumerate(PLAYER_NAMES)},
        valid={n: bool(vmask[i]) for i, n in enumerate(PLAYER_NAMES)},
        phi0=phi0, f=f, sum_phi=float(phi.sum()), sum_ctilde=float(c.sum()),
        residual_phi=float(residual), residual_ctilde=float(c_resid),
        spatial=dict(zip(["Self", "Route", "Transfer"], map(float, spatial))),
        temporal=dict(zip(["Old", "Mid", "Recent"], map(float, temporal))),
        boardings_baseline=b_base, boardings_pred=b_pred,
        negative_boardings=bool(neg), total_multiplier=mult, save_path=save_path)


### 10.2 Local Attribution Example


In [ ]:
import os, torch, numpy as np

BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"
CKPT     = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
OUTDIR   = f"{BASE_DIR}/results/FIXED_v6_no4683/local_explanations"

PAIRS       = [(195, 139)]
EXACT       = True
SHOW_CTILDE = True

for f in ("explain_target_node_phi_min", "load_checkpoint", "TA_META"):
    if f not in globals():
        raise RuntimeError(f"{f} is not defined - run its cell first")
os.makedirs(OUTDIR, exist_ok=True)

model, slots, hist = load_checkpoint(CKPT)
raw = torch.load(CKPT, map_location="cpu", weights_only=False)
if "test_data" not in raw:
    raise RuntimeError(f"{CKPT} was saved without test_loader - no embedded windows")
test_x, node_meta = raw["test_data"].float(), raw.get("labels")
n_test = int(test_x.shape[0])
del raw["state_dict"]                     
print(f"  arm A | test_data {tuple(test_x.shape)} | "
      f"{len(hist['valid_mse']) if hist else '?'} epochs"
      + ("" if node_meta else " | no labels, figures will say 'Node k'"))

ve = int(TA_META.get("valid_end", 0) or 0) or int(np.floor((1983 - 12) * 0.9))
if int(np.floor(0.9 * (ve + n_test))) != ve:
    raise RuntimeError(f"valid_end {ve} inconsistent with {n_test} test windows - "
                       f"re-run PrepareDataset on the matrix this model was trained on")
print(f"  valid_end {ve} CONFIRMED | test windows {n_test}")

results = []
for i, n in PAIRS:
    if not (0 <= i < n_test and 0 <= n < model.N):
        raise IndexError(f"({i}, {n}) outside [0,{n_test}) x [0,{model.N})")
    _, out = explain_target_node_phi_min(
        model=model, inputs=test_x[i:i + 1], target_node=n, slots=slots,
        start=ve + i, exact=EXACT, show_ctilde=SHOW_CTILDE, node_meta=node_meta,
        save_path=f"{OUTDIR}/armA_phi_node{n}_win{i}.png", close=len(PAIRS) > 1)
    results.append((i, out))

print(f"\n  {'win':>4}{'stop':>6}{'empty':>9}{'pred':>8}{'x(1+b)':>8}{'Self':>9}"
      f"{'Route':>9}{'Transf':>9}{'Old':>8}{'Mid':>8}{'Recent':>9}{'resid':>10}")
for i, r in results:
    s, t = r["spatial"], r["temporal"]
    print(f"  {i:>4}{r['target_node']:>6}{r['boardings_baseline']:>9.2f}"
          f"{r['boardings_pred']:>8.2f}{r['total_multiplier']:>8.2f}"
          f"{s['Self']:>+9.3f}{s['Route']:>+9.3f}{s['Transfer']:>+9.3f}"
          f"{t['Old']:>+8.3f}{t['Mid']:>+8.3f}{t['Recent']:>+9.3f}"
          f"{r['residual_phi']:>10.1e}")
print("\n  x(1+b) = exp(sum phi), a multiplier on (1 + boardings), not on boardings.")
print(f"  figures -> {OUTDIR}")


## 11. Intrinsic Spatiotemporal Contributions


In [ ]:
if True:
    import os as _os
    import numpy as _np, torch as _torch
    import matplotlib.pyplot as _plt

    _need = ["reference_branch_outs_target", "resolve_template", "get_player_specs",
             "get_valid_player_mask", "PLAYER_NAMES", "PLAYER_COLORS", "IDX_SELF",
             "IDX_ROUTE", "IDX_TRANSFER", "IDX_OLD", "IDX_MID", "IDX_RECENT",
             "load_checkpoint", "TA_META", "_node_caption"]
    _miss = [f for f in _need if f not in globals()]
    if _miss:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_miss))

    SPATIAL_COLORS = {
        "self":     "#4C78A8",   
        "route":    "#F28E2B",   
        "transfer": "#59A14F",   
    }

    TEMPORAL_COLORS = {
        "old":    "#9ECAE1",     
        "mid":    "#4292C6",     
        "recent": "#08519C",     
    }

    def _dc_bars(ax, vals, labels, colors, title, xlabel, ok, fs=8.5):
        y = _np.arange(len(labels))[::-1]

        b = ax.barh(
            y, vals,
            height=.62,
            edgecolor="white",
            linewidth=.7,
            color=[c if k else "#cfcfcf" for c, k in zip(colors, ok)]
        )

        ax.axvline(0, color="#333333", lw=.9)

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=fs)

        ax.set_xlabel(xlabel, fontsize=9.5)
        ax.tick_params(axis="x", labelsize=7.5)
        ax.tick_params(axis="y", length=0)

        ax.set_title(title, fontsize=9.5, fontweight="normal")

        ax.grid(axis="x", ls="--", lw=.6, alpha=.25)
        ax.set_axisbelow(True)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#777777")
        ax.spines["bottom"].set_color("#777777")
        ax.spines["left"].set_linewidth(.6)
        ax.spines["bottom"].set_linewidth(.6)

        s = max(1e-3, max(abs(v) for v in vals))

        for r, v, k in zip(b, vals, ok):
            ax.text(
                max(v, 0) + .035 * s,
                r.get_y() + r.get_height() / 2,
                f"{v:+.3f}" if k else "N/A",
                va="center",
                fontsize=7.2,
                color="#222222" if k else "#9a9a9a"
            )

        ax.set_xlim(
            min(0, min(vals)) - (.46 * s if min(vals) < 0 else .04 * s),
            max(0, max(vals)) + .46 * s
        )

        ax.set_ylim(-1.05, len(labels) - .45)

    @_torch.no_grad()
    def explain_target_node_intrinsic(model, inputs, target_node, slots, start,
                                      save_path="explanation_intrinsic.png",
                                      node_meta=None, tol=1e-3, close=False):
        if inputs.dim() != 3 or inputs.size(0) != 1:
            raise ValueError(f"inputs must be (1, 12, N); got {tuple(inputs.shape)}")

        dev, was_training = next(model.parameters()).device, model.training
        model.eval()

        tmpl = resolve_template(slots, int(start)).to(dev)
        if tmpl.dim() != 2 or tmpl.size(0) != inputs.size(1):
            raise ValueError(
                f"template must be ({inputs.size(1)}, N); {tuple(tmpl.shape)}"
            )

        dc_v, y, yref = reference_branch_outs_target(
            model, inputs.to(dev), tmpl, target_node
        )

        dc = dc_v.detach().cpu().numpy().astype(_np.float64)

        f  = float(y[0, 0, target_node].item())
        v0 = float(yref[0, 0, target_node].item())

        vm = get_valid_player_mask(
            get_player_specs(model, target_node)
        ).cpu().numpy().astype(bool)

        sp = _np.array([
            dc[i].sum()
            for i in (IDX_SELF, IDX_ROUTE, IDX_TRANSFER)
        ])

        tm = _np.array([
            dc[i].sum()
            for i in (IDX_OLD, IDX_MID, IDX_RECENT)
        ])

        res = abs(v0 + dc.sum() - f)

        if res > tol:
            raise RuntimeError(
                f"ADDITIVITY VIOLATED: |v0 + sum(dc) - f| = {res:.3e} "
                f"> {tol:.0e}  (v0 {v0:.6f}, sum {dc.sum():.6f}, "
                f"f {f:.6f}). slots/start do not match inputs."
            )

        if (~vm).any() and _np.abs(dc[~vm]).max() > 1e-9:
            raise RuntimeError(
                "an absent player carries a non-zero contribution"
            )

        if max(
            abs(sp.sum() - dc.sum()),
            abs(tm.sum() - dc.sum())
        ) > 1e-9:
            raise RuntimeError(
                "group totals do not re-sum to sum(dc)"
            )

        sp_ok = [
            bool(vm[i].any())
            for i in (IDX_SELF, IDX_ROUTE, IDX_TRANSFER)
        ]

        tm_ok = [
            bool(vm[i].any())
            for i in (IDX_OLD, IDX_MID, IDX_RECENT)
        ]

        b0, bf = float(_np.expm1(v0)), float(_np.expm1(f))

        print(
            f"  node {target_node} | start {int(start)} | "
            f"valid {int(vm.sum())}/9 | "
            f"v0 {v0:.6f} + sum(dc) {dc.sum():+.6f} "
            f"= {v0+dc.sum():.6f} vs f {f:.6f} "
            f"| residual {res:.1e} | "
            f"spatial {sp.sum():+.6f} temporal {tm.sum():+.6f}"
        )

        joint_labels = []
        joint_colors = []

        for n in PLAYER_NAMES:
            nl = n.lower()

            if "self" in nl:
                joint_colors.append(SPATIAL_COLORS["self"])
            elif "route" in nl:
                joint_colors.append(SPATIAL_COLORS["route"])
            elif "transfer" in nl:
                joint_colors.append(SPATIAL_COLORS["transfer"])
            else:
                joint_colors.append("#777777")

            lab = nl.replace("_", "–")
            lab = lab.replace("self", "Self")
            lab = lab.replace("route", "Route")
            lab = lab.replace("transfer", "Transfer")
            lab = lab.replace("old", "Old")
            lab = lab.replace("mid", "Mid")
            lab = lab.replace("recent", "Recent")

            joint_labels.append(lab)

        fig, ax = _plt.subplots(
            1, 3,
            figsize=(9.4, 3.55),
            gridspec_kw=dict(width_ratios=[1.55, 1, 1])
        )

        _dc_bars(
            ax[0],
            list(dc),
            joint_labels,
            joint_colors,
            "(a) Joint Spatio-Temporal\nIntrinsic Attribution",
            r"$\Delta c_i^{s,w}$",
            list(vm)
        )

        _dc_bars(
            ax[1],
            list(sp),
            ["Self", "Same-route", "Transfer"],
            [
                SPATIAL_COLORS["self"],
                SPATIAL_COLORS["route"],
                SPATIAL_COLORS["transfer"],
            ],
            "(b) Spatial Intrinsic\nGroup Attribution",
            r"$\sum_w\Delta c_i^{s,w}$",
            sp_ok,
            9
        )

        _dc_bars(
            ax[2],
            list(tm),
            ["Old", "Mid", "Recent"],
            [
                TEMPORAL_COLORS["old"],
                TEMPORAL_COLORS["mid"],
                TEMPORAL_COLORS["recent"],
            ],
            "(c) Temporal Intrinsic\nGroup Attribution",
            r"$\sum_s\Delta c_i^{s,w}$",
            tm_ok,
            9
        )


        fig.suptitle(
            rf"$v_i(\emptyset)={v0:.3f},\quad "
            rf"\sum_{{s,w}}\Delta c_i^{{s,w}}={dc.sum():.3f},\quad "
            rf"\hat{{x}}_{{t+1,i}}={f:.3f}$",
            fontsize=8.7,
            fontweight="bold",
            y=.995
        )

        fig.tight_layout()
        fig.subplots_adjust(top=.76)

        if _os.path.dirname(save_path):
            _os.makedirs(_os.path.dirname(save_path), exist_ok=True)

        fig.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight"
        )

        _plt.close(fig) if close else _plt.show()

        model.last_H_seq = model.last_branch_outs = None

        if was_training:
            model.train()

        return fig, dict(
            target_node=int(target_node),
            start=int(start),
            v_empty=v0,
            prediction=f,
            sum_delta_c=float(dc.sum()),
            residual=float(res),
            delta_c=dict(zip(PLAYER_NAMES, map(float, dc))),
            valid=dict(zip(PLAYER_NAMES, map(bool, vm))),
            spatial=dict(
                zip(["Self", "Route", "Transfer"], map(float, sp))
            ),
            temporal=dict(
                zip(["Old", "Mid", "Recent"], map(float, tm))
            ),
            spatial_ok=sp_ok,
            temporal_ok=tm_ok,
            boardings_baseline=b0,
            boardings_pred=bf,
            save_path=save_path
        )

    BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"

    CKPT = (
        f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/"
        f"ablation_A_guided_seed0.pt"
    )

    OUTDIR = (
        f"{BASE_DIR}/results/FIXED_v6_no4683/"
        f"local_explanations_intrinsic"
    )

    PAIRS = [(169, 262)]

    model, slots, _h = load_checkpoint(CKPT)

    raw = _torch.load(
        CKPT,
        map_location="cpu",
        weights_only=False
    )

    if "test_data" not in raw:
        raise RuntimeError(
            f"{CKPT} has no embedded test windows"
        )

    test_x, node_meta = (
        raw["test_data"].float(),
        raw.get("labels")
    )

    n_test = int(test_x.shape[0])

    del raw["state_dict"]

    ve = (
        int(TA_META.get("valid_end", 0) or 0)
        or int(_np.floor((1983 - 12) * .9))
    )

    if int(_np.floor(.9 * (ve + n_test))) != ve:
        raise RuntimeError(
            f"valid_end {ve} inconsistent with "
            f"{n_test} test windows"
        )

    print(
        f"  valid_end {ve} CONFIRMED | "
        f"test windows {n_test}"
    )

    for i, n in PAIRS:
        if not (
            0 <= i < n_test
            and 0 <= n < model.N
        ):
            raise IndexError(
                f"({i}, {n}) outside "
                f"[0,{n_test}) x [0,{model.N})"
            )

        explain_target_node_intrinsic(
            model=model,
            inputs=test_x[i:i + 1],
            target_node=n,
            slots=slots,
            start=ve + i,
            node_meta=node_meta,
            close=len(PAIRS) > 1,
            save_path=(
                f"{OUTDIR}/"
                f"armA_intrinsic_node{n}_win{i}.png"
            )
        )

    print(f"  figures -> {OUTDIR}")


### 11.1 Attribution Agreement and Efficiency


In [ ]:
from scipy.stats import spearmanr as _sp_scipy

def _spearman(a, b):
    if len(a) < 2:
        return float("nan")
    r = float(_sp_scipy(a, b)[0])
    return r if r == r else float("nan")

@torch.no_grad()
def cp_phi_agreement(model, test_x, baseline_template, n_pairs=30, seed=0, starts=None,
                     return_raw=False):
    device = next(model.parameters()).device
    model.eval()
    rng = random.Random(seed)
    S, N = test_x.size(0), model.N
    base = TA_META.get("valid_end", 0)
    pairs = [(rng.randrange(S), rng.randrange(N)) for _ in range(n_pairs)]
    sp, sign, cos, eff = [], [], [], []
    for s_i, n in pairs:
        s    = int(starts[s_i]) if starts is not None else base + s_i
        x    = test_x[s_i:s_i+1].to(device)
        tmpl = resolve_template(baseline_template, s).to(device)
        c_vec, _, _ = reference_branch_outs_target(model, x, tmpl, n)
        c   = c_vec.cpu().numpy()
        phi = estimate_group_shapley(model, x, n, tmpl, exact=True)
        phi = np.array([phi.get(p, 0.0) for p in PLAYER_NAMES])
        valid = get_valid_player_mask(get_player_specs(model, n)).numpy().astype(bool)
        if valid.sum() < 2: continue
        c, phi = c[valid], phi[valid]
        sp.append(_spearman(np.abs(c), np.abs(phi)))
        sign.append(float(np.mean(np.sign(c) == np.sign(phi))))
        cos.append(float(c @ phi / (np.linalg.norm(c)*np.linalg.norm(phi) + 1e-9)))
        eff.append(abs(float(c.sum() - phi.sum())))
    print(f"    cp_phi: mean |sum c~ - sum phi| = {np.mean(eff):.2e} "
          f"({'MATCHED' if np.mean(eff) < 1e-5 else 'MISMATCHED'})", flush=True)
    if return_raw:
        return (float(np.nanmean(sp)), float(np.nanmean(sign)), float(np.nanmean(cos)),
                dict(spearman=np.asarray(sp,   dtype=np.float64),
                     sign    =np.asarray(sign, dtype=np.float64),
                     cosine  =np.asarray(cos,  dtype=np.float64),
                     eff     =np.asarray(eff,  dtype=np.float64)))
    return (float(np.nanmean(sp)), float(np.nanmean(sign)), float(np.nanmean(cos)))

@torch.no_grad()
def efficiency_gap(model, test_x, baseline_template, n_samples=200, seed=0, starts=None):
    device = next(model.parameters()).device
    model.eval()
    rng  = random.Random(seed)
    idxs = [rng.randrange(test_x.size(0)) for _ in range(min(n_samples, test_x.size(0)))]
    base = TA_META.get("valid_end", 0)
    gaps, mags = [], []
    for i in idxs:
        s = int(starts[i]) if starts is not None else base + i
        c_tilde, y, v0 = reference_branch_outs(
            model, test_x[i:i+1].to(device), baseline_template, [s])
        delta = y - v0
        gaps.append((c_tilde.sum(dim=1) - delta).abs().mean().item())
        mags.append(delta.abs().mean().item())
    return float(np.mean(gaps)), float(np.mean(gaps) / (np.mean(mags) + 1e-9))

@torch.no_grad()
def _ab_rel_l2(model, te, slots, n_pairs, seed=0):
    dev = next(model.parameters()).device
    tx, tst = te.dataset.tensors[0], te.dataset.tensors[2]
    rng = random.Random(seed)
    vals, floored, clamped = [], 0, 0
    for _ in range(n_pairs):
        si, n = rng.randrange(tx.size(0)), rng.randrange(model.N)
        x    = tx[si:si + 1].to(dev)
        tmpl = resolve_template(slots, int(tst[si])).to(dev)
        c_vec, _, _ = reference_branch_outs_target(model, x, tmpl, n)
        phi_d = estimate_group_shapley(model, x, n, tmpl, exact=True)
        phi   = torch.tensor([phi_d.get(p, 0.0) for p in PLAYER_NAMES],
                             dtype=torch.float32, device=dev)
        vm = get_valid_player_mask(get_player_specs(model, n)).to(dev) > 0
        if int(vm.sum()) < 2 or float(phi[vm].abs().sum()) < 1e-8:
            continue
        raw_den = float((phi[vm] ** 2).sum())
        floored += raw_den < SHAP_DEN_FLOOR
        v = float(((c_vec[vm] - phi[vm]) ** 2).sum()) / max(raw_den, SHAP_DEN_FLOOR)
        clamped += v > SHAP_LOSS_CAP
        vals.append(min(v, SHAP_LOSS_CAP))
    v = np.asarray(vals)
    if v.size == 0:
        return dict(rel_l2_mean=np.nan, rel_l2_median=np.nan, rel_l2_sem=np.nan,
                    rel_l2_n=0, floor_pct=np.nan, clamp_pct=np.nan)
    return dict(rel_l2_mean=float(v.mean()), rel_l2_median=float(np.median(v)),
                rel_l2_sem=float(v.std(ddof=1) / np.sqrt(v.size)) if v.size > 1 else np.nan,
                rel_l2_n=v.size, floor_pct=100 * floored / v.size,
                clamp_pct=100 * clamped / v.size)


## 12. Fine-Grained Owen Attribution


### 12.1 Exact and Monte Carlo Owen Estimation


In [ ]:
import random as _random
import numpy as np
import torch

@torch.no_grad()
def node_level_scores(model, x_one, tmpl, target_node, eps=1e-12):
    specs = get_player_specs(model, target_node)
    x_ref = apply_player_mask(x_one, tmpl, set(), specs)

    model(x_one, return_branch_outs=True); H     = model.last_H_seq[0]
    model(x_ref, return_branch_outs=True); H_ref = model.last_H_seq[0]
    dev = (H - H_ref).norm(dim=-1)

    route_nodes    = torch.where(model.route_group_mask[target_node] > 0)[0].tolist()
    transfer_nodes = torch.where(model.transfer_group_mask[target_node] > 0)[0].tolist()
    involved = sorted({int(target_node)} | set(route_nodes) | set(transfer_nodes))

    node_raw = dev.mean(dim=0)
    node_w   = torch.zeros_like(node_raw)
    node_w[target_node] = 1.0
    for grp in (route_nodes, transfer_nodes):
        if not grp: continue
        idx = torch.tensor(grp, device=node_raw.device)
        v   = node_raw[idx]
        s   = float(v.sum())
        node_w[idx] = v / (s + eps) if s > eps else 1.0 / len(grp)

    inv     = torch.tensor(involved, device=dev.device)
    time_raw = dev[:, inv].mean(dim=1)
    time_w   = torch.zeros_like(time_raw)
    for w, ti in WINDOWS.items():
        idx = torch.tensor(ti, device=dev.device)
        v   = time_raw[idx]
        s   = float(v.sum())
        time_w[idx] = v / (s + eps) if s > eps else 1.0 / len(ti)

    model.last_H_seq = None; model.last_branch_outs = None
    return node_w, time_w, involved

def _involved_sets(model, n):
    n = int(n)
    route_nodes = torch.where(model.route_group_mask[n] > 0)[0].tolist()
    trans_nodes = torch.where(model.transfer_group_mask[n] > 0)[0].tolist()
    seen = {n} | set(route_nodes)
    trans_nodes = [j for j in trans_nodes if j not in seen]
    return [n], route_nodes, trans_nodes

def apply_unit_mask(x_row, tmpl, masked_units, mask_kind, involved, mask_mode):
    x_m = x_row.clone() if x_row.dim() == 3 else x_row.unsqueeze(0).clone()
    if mask_kind == "nodes":
        nodes = sorted({int(j) for u in masked_units for j in u})
        if nodes:
            x_m[:, :, nodes] = 0.0 if mask_mode == "zero" else tmpl[:, nodes].unsqueeze(0)
    else:
        times = sorted({int(t) for u in masked_units for t in u})
        inv = torch.tensor(sorted(set(int(j) for j in involved)),
                           dtype=torch.long, device=x_m.device)
        for t in times:
            x_m[:, t, inv] = 0.0 if mask_mode == "zero" else tmpl[t, inv].unsqueeze(0)
    return x_m

def order_units(scores, ordering, rng):
    if ordering == "model":
        return np.argsort(-scores, kind="stable")
    if ordering == "reverse":
        return np.argsort(scores, kind="stable")
    if ordering == "random":
        idx = list(range(len(scores)))
        rng.shuffle(idx)
        return np.asarray(idx)
    raise ValueError(ordering)

def spatial_partition(model, n):
    self_n, route_n, trans_n = _involved_sets(model, n)
    return [u for u in (list(self_n), list(route_n), list(trans_n)) if len(u) > 0]

def temporal_partition():
    return [list(WINDOWS[w]) for w in ("old", "mid", "recent")]

def _mask_complement(x_row, tmpl, present_atoms, all_atoms, kind,
                     involved, mask_mode="timeaware"):
    absent = [a for a in all_atoms if a not in present_atoms]
    return apply_unit_mask(x_row, tmpl, [[a] for a in absent], kind,
                           involved, mask_mode)

@torch.no_grad()
def estimate_owen_values(model, x_row, tmpl, target_node, mode,
                         n_perm=200, mask_mode="timeaware",
                         eval_batch=256, seed=0, return_group=True):
    device = next(model.parameters()).device
    n = int(target_node)
    self_n, route_n, trans_n = _involved_sets(model, n)
    involved = self_n + route_n + trans_n

    if mode == "spatial":
        unions, kind = spatial_partition(model, n), "nodes"
    elif mode == "temporal":
        unions, kind = temporal_partition(), "times"
    else:
        raise ValueError(mode)

    atoms = [a for u in unions for a in u]
    A = len(atoms)
    if A == 0:
        return {}, {}, {}

    owen = {a: 0.0 for a in atoms}
    group = {gi: 0.0 for gi in range(len(unions))}
    atom_union = {a: gi for gi, u in enumerate(unions) for a in u}
    rng = _random.Random(seed)

    perms_per_chunk = max(1, (eval_batch * 4) // (A + 1))

    p = 0
    while p < n_perm:
        chunk = min(perms_per_chunk, n_perm - p)
        orders, graphs = [], []
        for _ in range(chunk):
            u_order = list(range(len(unions)))
            rng.shuffle(u_order)
            order = []
            for gi in u_order:
                members = list(unions[gi])
                rng.shuffle(members)
                order.extend(members)
            orders.append(order)
            present = []
            graphs.append(_mask_complement(x_row, tmpl, present, atoms, kind,
                                           involved, mask_mode))
            for a in order:
                present = present + [a]
                graphs.append(_mask_complement(x_row, tmpl, present, atoms, kind,
                                               involved, mask_mode))

        vals = []
        big = torch.cat(graphs, dim=0)
        for s in range(0, big.size(0), eval_batch):
            yb = model(big[s:s + eval_batch].to(device))
            vals.extend(float(yb[j, 0, n].item()) for j in range(yb.size(0)))
            del yb
        del big, graphs

        stride = A + 1
        for k, order in enumerate(orders):
            base = k * stride
            for t, a in enumerate(order):
                delta = vals[base + t + 1] - vals[base + t]
                owen[a] += delta
                group[atom_union[a]] += delta
        p += chunk

    for a in owen:
        owen[a] /= float(n_perm)
    for gi in group:
        group[gi] /= float(n_perm)

    meta = dict(atoms=atoms, unions=unions, kind=kind, involved=involved)
    if return_group:
        x_full = x_row.to(device)
        x_empty = _mask_complement(x_row, tmpl, [], atoms, kind, involved, mask_mode)
        meta["f_full"] = float(model(x_full)[0, 0, n].item())
        meta["f_empty"] = float(model(x_empty.to(device))[0, 0, n].item())
    return owen, group, meta


import math as _math
from itertools import combinations as _combos, chain as _chain


def _all_subsets(xs):
    for r in range(len(xs) + 1):
        for c in _combos(xs, r):
            yield frozenset(c)


def _unions_for(model, n, mode):
    if mode == "spatial":
        return spatial_partition(model, n), "nodes"
    if mode == "temporal":
        return temporal_partition(), "times"
    raise ValueError(mode)


@torch.no_grad()
def _eval_coalitions(model, x_row, tmpl, n, coalitions, atoms, kind,
                     involved, mask_mode, eval_batch=256):
    device = next(model.parameters()).device
    coalitions = list(coalitions)
    out = {}
    for s in range(0, len(coalitions), eval_batch):
        chunk = coalitions[s:s + eval_batch]
        xb = torch.cat([_mask_complement(x_row, tmpl, list(c), atoms, kind,
                                         involved, mask_mode) for c in chunk], dim=0)
        yb = model(xb.to(device))
        for j, c in enumerate(chunk):
            out[c] = float(yb[j, 0, n].item())
        del xb, yb
    model.last_H_seq = None; model.last_branch_outs = None
    return out


@torch.no_grad()
def exact_owen_values(model, x_row, tmpl, target_node, mode,
                      mask_mode="timeaware", eval_batch=256, max_union=12):
    n = int(target_node)
    self_n, route_n, trans_n = _involved_sets(model, n)
    involved = self_n + route_n + trans_n
    unions, kind = _unions_for(model, n, mode)
    atoms = [a for u in unions for a in u]
    if not atoms:
        return {}, {}, {}
    big = max(len(u) for u in unions)
    if big > max_union:
        raise ValueError(
            f"exact Owen infeasible: largest union has {big} atoms "
            f"(> max_union={max_union}); cost ~ 2^{big}. Use "
            f"estimate_owen_values (permutation sampling) for this mode.")

    M = len(unions)
    need = set()
    for k in range(M):
        others = [i for i in range(M) if i != k]
        for R in _all_subsets(others):
            base = frozenset(_chain.from_iterable(unions[j] for j in R))
            for T in _all_subsets(unions[k]):
                need.add(base | T)
    need.add(frozenset()); need.add(frozenset(atoms))
    vals = _eval_coalitions(model, x_row, tmpl, n, need, atoms, kind,
                            involved, mask_mode, eval_batch)

    owen = {}
    group = {gi: 0.0 for gi in range(M)}
    for k, Sk in enumerate(unions):
        others = [i for i in range(M) if i != k]
        nk = len(Sk)
        for i in Sk:
            tot = 0.0
            for R in _all_subsets(others):
                base = frozenset(_chain.from_iterable(unions[j] for j in R))
                wR = _math.factorial(len(R)) * _math.factorial(M - len(R) - 1) \
                     / _math.factorial(M)
                for T in _all_subsets([a for a in Sk if a != i]):
                    wT = _math.factorial(len(T)) * _math.factorial(nk - len(T) - 1) \
                         / _math.factorial(nk)
                    tot += wR * wT * (vals[base | T | {i}] - vals[base | T])
            owen[i] = tot
            group[k] += tot

    meta = dict(atoms=atoms, unions=unions, kind=kind, involved=involved,
                f_full=vals[frozenset(atoms)], f_empty=vals[frozenset()],
                n_coalitions=len(need), exact=True)
    return owen, group, meta


@torch.no_grad()
def exact_quotient_shapley(model, x_row, tmpl, target_node, mode,
                           mask_mode="timeaware"):
    n = int(target_node)
    self_n, route_n, trans_n = _involved_sets(model, n)
    involved = self_n + route_n + trans_n
    unions, kind = _unions_for(model, n, mode)
    atoms = [a for u in unions for a in u]
    if not atoms:
        return {}, {}
    M = len(unions)
    need = {frozenset(_chain.from_iterable(unions[j] for j in R))
            for R in _all_subsets(list(range(M)))}
    vals = _eval_coalitions(model, x_row, tmpl, n, need, atoms, kind,
                            involved, mask_mode)
    sh = {}
    for k in range(M):
        others = [i for i in range(M) if i != k]
        tot = 0.0
        for R in _all_subsets(others):
            w = _math.factorial(len(R)) * _math.factorial(M - len(R) - 1) \
                / _math.factorial(M)
            base = frozenset(_chain.from_iterable(unions[j] for j in R))
            withk = base | frozenset(unions[k])
            tot += w * (vals[withk] - vals[base])
        sh[k] = tot
    return sh, dict(f_full=vals[frozenset(atoms)], f_empty=vals[frozenset()])


### 12.2 Spatial Owen Values


In [ ]:
if True:
    import os
    from math import factorial
    import numpy as np, pandas as pd, torch
    import matplotlib.pyplot as plt

    BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"
    CKPT     = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
    OUT      = f"{BASE_DIR}/results/FIXED_v6_no4683/owen_spatial"

    WINDOW_I    = 169
    TARGET_NODE = 262


    TOP_N = 6
    BATCH = 256
    MAX_ATOMS_WARN = 16

    _need = ["load_checkpoint", "resolve_template", "get_player_specs",
             "apply_player_mask", "estimate_group_shapley", "PLAYER_NAMES", "TA_META"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_missing))
    os.makedirs(OUT, exist_ok=True)
    print("  pre-flight OK")

    def _ow_values(n_atoms, groups, v):
        m     = len(groups)
        gmask = [sum(1 << j for j in g) for g in groups]
        omega, phi_q = [0.0] * n_atoms, [0.0] * m
        for k, B in enumerate(groups):
            b      = len(B)
            others = [i for i in range(m) if i != k]
            for qbits in range(1 << (m - 1)):
                Q     = [others[i] for i in range(m - 1) if qbits >> i & 1]
                base  = 0
                for q in Q:
                    base |= gmask[q]
                w_out = factorial(len(Q)) * factorial(m - 1 - len(Q)) / factorial(m)
                phi_q[k] += w_out * (v(base | gmask[k]) - v(base))
                if b == 0:
                    continue
                val = {}
                for sbits in range(1 << b):
                    mk = base
                    for i in range(b):
                        if sbits >> i & 1:
                            mk |= 1 << B[i]
                    val[sbits] = v(mk)
                for i, j in enumerate(B):
                    rest = [t for t in range(b) if t != i]
                    for sbits in range(1 << (b - 1)):
                        s = 0
                        for t in range(b - 1):
                            if sbits >> t & 1:
                                s |= 1 << rest[t]
                        sz   = bin(s).count("1")
                        w_in = factorial(sz) * factorial(b - 1 - sz) / factorial(b)
                        omega[j] += w_out * w_in * (val[s | (1 << i)] - val[s])
        return omega, phi_q

    owen_model, owen_slots, _ = load_checkpoint(CKPT)
    _raw   = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = _raw["test_data"].float()
    lab    = _raw.get("labels")
    n_test = int(test_x.shape[0])
    _raw.pop("state_dict", None)
    dev    = next(owen_model.parameters()).device
    _was_tr = owen_model.training
    owen_model.eval()

    _ve = int(TA_META.get("valid_end", 0) or 0)
    if _ve <= 0:
        _ve = int(np.floor((1983 - 12 - 1 + 1) * 0.9))
    if int(np.floor(0.9 * (_ve + n_test))) != _ve:
        raise RuntimeError(f"valid_end {_ve} inconsistent with {n_test} test windows")
    if not (0 <= WINDOW_I < n_test):
        raise IndexError(f"WINDOW_I {WINDOW_I} outside [0, {n_test-1}]")
    _start = _ve + WINDOW_I

    _specs = get_player_specs(owen_model, TARGET_NODE)
    _route = list(_specs["route_recent"]["nodes"])
    _trans = list(_specs["transfer_recent"]["nodes"])
    if TARGET_NODE in _route or TARGET_NODE in _trans:
        raise RuntimeError("target node appears in its own neighbour set -- groups overlap")
    if set(_route) & set(_trans):
        raise RuntimeError("route and transfer sets overlap -- not a partition")

    owen_atoms  = [TARGET_NODE] + _route + _trans
    owen_groups = [[0],
                   list(range(1, 1 + len(_route))),
                   list(range(1 + len(_route), 1 + len(_route) + len(_trans)))]
    _GN = ["self", "route", "transfer"]
    _nA = len(owen_atoms)
    print(f"  target node {TARGET_NODE} | window {WINDOW_I} -> absolute start {_start}")
    print(f"  atoms: 1 self + {len(_route)} route + {len(_trans)} transfer = {_nA}")
    if max(len(g) for g in owen_groups) > MAX_ATOMS_WARN:
        raise RuntimeError(
            f"largest group has {max(len(g) for g in owen_groups)} atoms -> "
            f"2^2 x 2^{max(len(g) for g in owen_groups)} evaluations. Pick a target with a "
            f"smaller route group -- routes 4543 and 4571 have 15 stops each, "
            f"so their nodes have k_route = 14, the minimum in this network.")

    _x    = test_x[WINDOW_I:WINDOW_I + 1].to(dev)
    _tmpl = resolve_template(owen_slots, _start).to(dev)

    _cache = {}

    _x_ref9 = apply_player_mask(_x, _tmpl, set(), _specs)
    _x_ref_owen = _x.clone()
    _x_ref_owen[0][:, owen_atoms] = _tmpl[:, owen_atoms]
    _dmax = float((_x_ref9 - _x_ref_owen).abs().max().item())
    print(f"  EMPTY-COALITION IDENTITY: max|x_ref(9-player) - x_ref(19-atom)| = {_dmax:.3e}")
    if _dmax > 1e-6:
        raise RuntimeError(
            "the two games do NOT mask the same cells at the empty coalition "
            f"(max deviation {_dmax:.3e}); the cross-game total check would be "
            "comparing different quantities")
    del _x_ref9, _x_ref_owen

    def _v_eval(masks):
        todo = [m for m in dict.fromkeys(masks) if m not in _cache]
        for s in range(0, len(todo), BATCH):
            chunk = todo[s:s + BATCH]
            xb = _x.expand(len(chunk), -1, -1).clone()
            for bi, mk in enumerate(chunk):
                drop = [owen_atoms[j] for j in range(_nA) if not (mk >> j & 1)]
                if drop:
                    xb[bi][:, drop] = _tmpl[:, drop]
            with torch.no_grad():
                pr = owen_model(xb)
            for bi, mk in enumerate(chunk):
                _cache[mk] = float(pr[bi, 0, TARGET_NODE].item())
            owen_model.last_H_seq = None
            owen_model.last_branch_outs = None
            del xb, pr
            if (s // BATCH) % 40 == 0 and s:
                print(f"    {s + len(chunk):,}/{len(todo):,} evaluated", flush=True)

    _wanted = []
    _ow_values(_nA, owen_groups, lambda mk: (_wanted.append(mk), 0.0)[1])
    _uniq = list(dict.fromkeys(_wanted))
    print(f"  {len(_wanted):,} coalition requests -> {len(_uniq):,} distinct forwards")
    import time as _t
    _t0 = _t.time()
    _v_eval(_uniq)
    print(f"  evaluated in {(_t.time()-_t0)/60:.1f} min "
          f"({(_t.time()-_t0)/max(len(_uniq),1)*1000:.2f} ms per forward)")

    owen_omega, owen_phi_s = _ow_values(_nA, owen_groups, lambda mk: _cache[mk])

    _full = (1 << _nA) - 1
    _f, _v0 = _cache[_full], _cache[0]
    print("\n  " + "=" * 74)
    print("  EXACTNESS CHECKS")
    print("  " + "=" * 74)
    _ok = True
    for k, g in enumerate(owen_groups):
        _s, _p = sum(owen_omega[j] for j in g), owen_phi_s[k]
        _d = abs(_s - _p); _ok &= _d < 1e-6
        print(f"    sum Omega over B_{_GN[k]:<9} = {_s:+.8f}   Phi_{_GN[k]}^s = {_p:+.8f}   "
              f"|diff| {_d:.2e}")
    _d = abs(sum(owen_omega) - (_f - _v0)); _ok &= _d < 1e-6
    print(f"    efficiency: sum Omega = {sum(owen_omega):+.8f}   f - v(empty) = "
          f"{_f - _v0:+.8f}   |diff| {_d:.2e}")
    _d = abs(sum(owen_phi_s) - (_f - _v0)); _ok &= _d < 1e-6
    print(f"    quotient efficiency: sum Phi^s = {sum(owen_phi_s):+.8f}   |diff| {_d:.2e}")
    if not _ok:
        raise RuntimeError("an Owen identity failed -- do not use these numbers")
    print(f"\n    f = {_f:+.6f}   v(empty) = {_v0:+.6f}   (log1p space)")

    _phi9 = estimate_group_shapley(owen_model, _x, TARGET_NODE, _tmpl, exact=True)
    if isinstance(_phi9, dict):
        _sum9 = float(sum(_phi9.values()))
    elif isinstance(_phi9, (tuple, list)) and isinstance(_phi9[0], dict):
        _sum9 = float(sum(_phi9[0].values()))
    else:
        _sum9 = float(np.asarray(_phi9, dtype=float).ravel()[:len(PLAYER_NAMES)].sum())
    _d = abs(_sum9 - (_f - _v0))
    print(f"    CROSS-GAME: sum(phi) over the 9 players = {_sum9:+.8f}")
    print(f"                f - v(empty) from this game = {_f - _v0:+.8f}   |diff| {_d:.2e}")
    if _d > 1e-5:
        raise RuntimeError(
            f"CROSS-GAME CHECK FAILED by {_d:.3e}. The two games do not share an "
            f"empty coalition, so one of them masks a different cell set. Do not "
            f"present these Owen values next to the nine-player figure.")
    print(f"                -> the two games agree on the total. Only the split differs.")

    def _cap(n):
        return (f"stop {lab['stop_ids'][n]} / R{lab['route_ids'][n]}"
                if lab else f"node {n}")
    _fam = ["self"] * 1 + ["route"] * len(_route) + ["transfer"] * len(_trans)
    owen_tab = pd.DataFrame({
        "atom_index": range(_nA),
        "node": owen_atoms,
        "family": _fam,
        "stop_id": [lab["stop_ids"][n] if lab else -1 for n in owen_atoms],
        "route_id": [lab["route_ids"][n] if lab else -1 for n in owen_atoms],
        "owen": owen_omega,
    })
    owen_tab["abs_owen"] = owen_tab["owen"].abs()
    owen_tab["abs_share_of_group_pct"] = owen_tab.groupby("family")["abs_owen"].transform(
        lambda s: 100 * s / s.sum() if s.sum() > 0 else np.nan)
    print("\n  ALL 19 OWEN VALUES (signed, log1p space), sorted within family by |Omega|")
    for fam in ["self", "route", "transfer"]:
        sub = owen_tab[owen_tab.family == fam].sort_values("abs_owen", ascending=False)
        print(f"    {fam.upper():<9} Phi^s = {owen_phi_s[_GN.index(fam)]:+.5f}")
        for _, r in sub.iterrows():
            print(f"       {_cap(int(r.node)):<28} node {int(r.node):>3}   "
                  f"Omega {r.owen:+.5f}")
    owen_tab.to_csv(f"{OUT}/owen_node{TARGET_NODE}_win{WINDOW_I}.csv", index=False)

    _COL = {"self": "#4C78A8", "route": "#F28E2B", "transfer": "#59A14F"}

    def _rowlab(n):
        return (f"{lab['stop_ids'][n]} / Route {lab['route_ids'][n]}"
                if lab else f"Node {n}")

    _r = owen_tab[owen_tab.family == "route"].sort_values("abs_owen", ascending=False)
    _named, _rest = _r.head(TOP_N), _r.iloc[TOP_N:]
    _tr = owen_tab[owen_tab.family == "transfer"].sort_values("abs_owen", ascending=False)
    _sf = owen_tab[owen_tab.family == "self"]          
    if len(_sf) != 1:
        raise RuntimeError(f"expected exactly one self atom, found {len(_sf)}")

    _sections = [
        ("Self", "self",
         [(f"Target: {_rowlab(TARGET_NODE)}", float(_sf.owen.iloc[0]))]),
        ("Same-route stops", "route",
         [(_rowlab(int(x.node)), float(x.owen)) for _, x in _named.iterrows()]
         + ([(f"Remaining same-route stops ({len(_rest)})",
              float(_rest.owen.sum()))] if len(_rest) else [])),
        ("Transfer stops", "transfer",
         [(_rowlab(int(x.node)), float(x.owen)) for _, x in _tr.iterrows()]),
    ]

    _GAP = 1.0
    _lab, _val, _col, _ypos, _cuts, _heads = [], [], [], [], [], []
    _y = 0.0
    for _si, (_title, _fam, _items) in enumerate(_sections):
        if not _items:
            continue
        if _si:
            _cuts.append(_y + (1 - _GAP) / 2)
            _y -= _GAP
        _heads.append((_title, _y + 0.72))
        for _n, _v in _items:
            _lab.append(_n); _val.append(_v); _col.append(_COL[_fam]); _ypos.append(_y)
            _y -= 1.0

    fig, ax = plt.subplots(figsize=(7.2, 0.30 * len(_lab) + 1.9))
    ax.barh(_ypos, _val, height=.68, color=_col, edgecolor="white", linewidth=.6)
    ax.axvline(0, color="#333333", lw=.9)
    for _c in _cuts:
        ax.axhline(_c, color="#d0d0d0", lw=.7)
    ax.set_yticks(_ypos); ax.set_yticklabels(_lab, fontsize=8)
    ax.tick_params(axis="y", length=0); ax.tick_params(axis="x", labelsize=7.5)

    _sp = max(abs(v) for v in _val)
    _lo, _hi = min(0, min(_val)), max(0, max(_val))
    for _yv, _v in zip(_ypos, _val):
        ax.text(_v + (.03 if _v >= 0 else -.03) * _sp, _yv, f"{_v:+.3f}",
                va="center", ha="left" if _v >= 0 else "right", fontsize=7.2,
                color="#222222")
    ax.set_xlim(_lo - (.34 * _sp if _lo < 0 else .04 * _sp),
                _hi + .34 * _sp)
    ax.set_ylim(min(_ypos) - .8, max(_ypos) + 1.25)

    for _t, _yh in _heads:
        ax.text(.004, _yh, _t, transform=ax.get_yaxis_transform(), ha="left",
                va="center", fontsize=8, fontweight="bold", color="#555555")

    ax.set_xlabel(r"Signed Owen value  $\Omega$", fontsize=9)
    ax.grid(axis="x", ls="--", lw=.6, alpha=.25); ax.set_axisbelow(True)
    for _s in ("top", "right"):
        ax.spines[_s].set_visible(False)
    for _s in ("left", "bottom"):
        ax.spines[_s].set_color("#777777"); ax.spines[_s].set_linewidth(.6)

    fig.suptitle("(a) Fine-Grained Spatial Owen Attribution",
                 fontsize=10.5, fontweight="normal", y=.985)

    fig.text(.5, .018,
             r"$\sum_{a \in G_k}\Omega_{i,a}=\Phi^s_{i,G_k}$:    "
             rf"$\Phi^s_{{\mathrm{{Self}}}}$ = {owen_phi_s[0]:+.3f}" + "     "
             f"$\\Phi^s_{{\\mathrm{{Same\u2011route}}}}$ = {owen_phi_s[1]:+.3f}" + "     "
             rf"$\Phi^s_{{\mathrm{{Transfer}}}}$ = {owen_phi_s[2]:+.3f}",
             ha="center", va="top", fontsize=8.5)

    fig.tight_layout(rect=[0, .055, 1, .955], pad=.4)
    fig.savefig(f"{OUT}/owen_spatial_node{TARGET_NODE}_win{WINDOW_I}.png",
                dpi=600, bbox_inches="tight")
    plt.close(fig)

    if _was_tr:
        owen_model.train()
    print(f"\n  figure -> {OUT}/owen_spatial_node{TARGET_NODE}_win{WINDOW_I}.png")
    print(f"  data   -> {OUT}/owen_node{TARGET_NODE}_win{WINDOW_I}.csv  (all {_nA} atoms)")


### 12.3 Temporal Owen Values


In [ ]:
if True:
    import os
    from math import factorial
    import numpy as np, pandas as pd, torch
    import matplotlib.pyplot as plt

    BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"
    CKPT     = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
    OUT      = f"{BASE_DIR}/results/FIXED_v6_no4683/owen_temporal"

    WINDOW_I    = 169
    TARGET_NODE = 262

    BIN_MIN     = 10

    _need = ["load_checkpoint", "resolve_template", "get_player_specs",
             "apply_player_mask", "estimate_group_shapley", "WINDOWS",
             "PLAYER_NAMES", "TA_META"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_missing))
    os.makedirs(OUT, exist_ok=True)
    print("  pre-flight OK")

    def _owt_values(n_atoms, groups, v):
        m     = len(groups)
        gmask = [sum(1 << j for j in g) for g in groups]
        omega, phi_q = [0.0] * n_atoms, [0.0] * m
        for k, B in enumerate(groups):
            b      = len(B)
            others = [i for i in range(m) if i != k]
            for qbits in range(1 << (m - 1)):
                Q     = [others[i] for i in range(m - 1) if qbits >> i & 1]
                base  = 0
                for q in Q:
                    base |= gmask[q]
                w_out = factorial(len(Q)) * factorial(m - 1 - len(Q)) / factorial(m)
                phi_q[k] += w_out * (v(base | gmask[k]) - v(base))
                if b == 0:
                    continue
                val = {}
                for sbits in range(1 << b):
                    mk = base
                    for i in range(b):
                        if sbits >> i & 1:
                            mk |= 1 << B[i]
                    val[sbits] = v(mk)
                for i, j in enumerate(B):
                    rest = [t for t in range(b) if t != i]
                    for sbits in range(1 << (b - 1)):
                        s = 0
                        for t in range(b - 1):
                            if sbits >> t & 1:
                                s |= 1 << rest[t]
                        sz   = bin(s).count("1")
                        w_in = factorial(sz) * factorial(b - 1 - sz) / factorial(b)
                        omega[j] += w_out * w_in * (val[s | (1 << i)] - val[s])
        return omega, phi_q

    owt_model, owt_slots, _ = load_checkpoint(CKPT)
    _raw   = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = _raw["test_data"].float()
    lab    = _raw.get("labels")
    n_test = int(test_x.shape[0])
    _raw.pop("state_dict", None)
    dev     = next(owt_model.parameters()).device
    _was_tr = owt_model.training
    owt_model.eval()

    _ve = int(TA_META.get("valid_end", 0) or 0)
    if _ve <= 0:
        _ve = int(np.floor((1983 - 12 - 1 + 1) * 0.9))
    if int(np.floor(0.9 * (_ve + n_test))) != _ve:
        raise RuntimeError(f"valid_end {_ve} inconsistent with {n_test} test windows")
    if not (0 <= WINDOW_I < n_test):
        raise IndexError(f"WINDOW_I {WINDOW_I} outside [0, {n_test-1}]")
    _start = _ve + WINDOW_I

    _specs = get_player_specs(owt_model, TARGET_NODE)
    _route = list(_specs["route_recent"]["nodes"])
    _trans = list(_specs["transfer_recent"]["nodes"])
    if TARGET_NODE in _route or TARGET_NODE in _trans:
        raise RuntimeError("target node appears in its own route/transfer set")
    if set(_route) & set(_trans):
        raise RuntimeError(
            f"route and transfer sets overlap on {sorted(set(_route) & set(_trans))} "
            f"-- not a partition, so a masked cell would be double-counted")
    owt_nodes = [TARGET_NODE] + _route + _trans
    if len(set(owt_nodes)) != len(owt_nodes):
        raise RuntimeError("duplicate nodes in the temporal context set")
    print(f"  target node {TARGET_NODE} | window {WINDOW_I} -> absolute start {_start}")
    print(f"  masking is restricted to the SAME {len(owt_nodes)} nodes as the spatial game "
          f"(1 self + {len(_route)} route + {len(_trans)} transfer)")

    _T = int(test_x.shape[1])
    owt_groups = [list(WINDOWS["old"]), list(WINDOWS["mid"]), list(WINDOWS["recent"])]
    _GN = ["old", "mid", "recent"]
    _flat = sorted(t for g in owt_groups for t in g)
    if _flat != list(range(_T)):
        raise RuntimeError(f"WINDOWS do not partition the {_T} timesteps: got {_flat}")
    print(f"  atoms: {_T} timesteps | groups "
          + " ".join(f"{n}={g}" for n, g in zip(_GN, owt_groups)))

    _x    = test_x[WINDOW_I:WINDOW_I + 1].to(dev)
    _tmpl = resolve_template(owt_slots, _start).to(dev)

    _cache = {}

    _x_ref9 = apply_player_mask(_x, _tmpl, set(), _specs)
    _x_ref_owt = _x.clone()
    _x_ref_owt[0][:, owt_nodes] = _tmpl[:, owt_nodes]
    _dmax = float((_x_ref9 - _x_ref_owt).abs().max().item())
    print(f"  EMPTY-COALITION IDENTITY: max|x_ref(9-player) - x_ref(temporal)| = {_dmax:.3e}")
    if _dmax > 1e-6:
        raise RuntimeError(
            f"the temporal game does not share an empty coalition with the "
            f"nine-player game (max deviation {_dmax:.3e}); the total check below "
            f"would compare different quantities")
    del _x_ref9, _x_ref_owt

    def _v_eval(masks):
        todo = [m for m in dict.fromkeys(masks) if m not in _cache]
        for s in range(0, len(todo), 256):
            chunk = todo[s:s + 256]
            xb = _x.expand(len(chunk), -1, -1).clone()
            for bi, mk in enumerate(chunk):
                drop = [t for t in range(_T) if not (mk >> t & 1)]
                for t in drop:
                    xb[bi][t, owt_nodes] = _tmpl[t, owt_nodes]
            with torch.no_grad():
                pr = owt_model(xb)
            for bi, mk in enumerate(chunk):
                _cache[mk] = float(pr[bi, 0, TARGET_NODE].item())
            owt_model.last_H_seq = None
            owt_model.last_branch_outs = None
            del xb, pr

    _wanted = []
    _owt_values(_T, owt_groups, lambda mk: (_wanted.append(mk), 0.0)[1])
    _uniq = list(dict.fromkeys(_wanted))
    print(f"  {len(_wanted)} coalition requests -> {len(_uniq)} distinct forwards")
    import time as _t
    _t0 = _t.time()
    _v_eval(_uniq)
    print(f"  evaluated in {_t.time()-_t0:.1f} s")

    owt_omega, owt_phi_t = _owt_values(_T, owt_groups, lambda mk: _cache[mk])

    _full = (1 << _T) - 1
    _f, _v0 = _cache[_full], _cache[0]
    print("\n  " + "=" * 74)
    print("  EXACTNESS CHECKS")
    print("  " + "=" * 74)
    _ok = True
    for k, g in enumerate(owt_groups):
        _s, _p = sum(owt_omega[j] for j in g), owt_phi_t[k]
        _d = abs(_s - _p); _ok &= _d < 1e-6
        print(f"    sum Omega over G_{_GN[k]:<7} = {_s:+.8f}   Phi_{_GN[k]}^t = {_p:+.8f}   "
              f"|diff| {_d:.2e}")
    _d = abs(sum(owt_omega) - (_f - _v0)); _ok &= _d < 1e-6
    print(f"    efficiency: sum Omega = {sum(owt_omega):+.8f}   f - v(empty) = "
          f"{_f - _v0:+.8f}   |diff| {_d:.2e}")
    _d = abs(sum(owt_phi_t) - (_f - _v0)); _ok &= _d < 1e-6
    print(f"    quotient efficiency: sum Phi^t = {sum(owt_phi_t):+.8f}   |diff| {_d:.2e}")
    if not _ok:
        raise RuntimeError("an Owen identity failed -- do not use these numbers")

    _phi9 = estimate_group_shapley(owt_model, _x, TARGET_NODE, _tmpl, exact=True)
    if isinstance(_phi9, dict):
        _sum9 = float(sum(_phi9.values()))
    elif isinstance(_phi9, (tuple, list)) and isinstance(_phi9[0], dict):
        _sum9 = float(sum(_phi9[0].values()))
    else:
        _sum9 = float(np.asarray(_phi9, dtype=float).ravel()[:len(PLAYER_NAMES)].sum())
    _d = abs(_sum9 - (_f - _v0))
    print(f"\n    CROSS-GAME: sum(phi) over the 9 players = {_sum9:+.8f}")
    print(f"                f - v(empty), temporal game  = {_f - _v0:+.8f}   |diff| {_d:.2e}")
    if _d > 1e-5:
        raise RuntimeError(f"CROSS-GAME CHECK FAILED by {_d:.3e}")
    print(f"                -> three games (9-player, spatial, temporal) agree on the total.")

    _tlab = [f"t-{_T - 1 - t}" if t < _T - 1 else "t" for t in range(_T)]
    _fam = [next(n for n, g in zip(_GN, owt_groups) if t in g) for t in range(_T)]
    owt_tab = pd.DataFrame({
        "timestep": range(_T),
        "lag": _tlab,
        "minutes_before": [(_T - t) * BIN_MIN for t in range(_T)],
        "window": _fam,
        "owen": owt_omega,
    })
    owt_tab["abs_owen"] = owt_tab["owen"].abs()
    owt_tab["abs_share_of_window_pct"] = owt_tab.groupby("window")["abs_owen"].transform(
        lambda s: 100 * s / s.sum() if s.sum() > 0 else np.nan)
    print(f"\n  ALL {_T} TEMPORAL OWEN VALUES (signed, log1p space)")
    for w in _GN:
        sub = owt_tab[owt_tab.window == w]
        print(f"    {w.upper():<7} Phi^t = {owt_phi_t[_GN.index(w)]:+.5f}")
        for _, r in sub.iterrows():
            print(f"       {r.lag:<5} ({int(r.minutes_before):>3} min before t+1)   "
                  f"Omega {r.owen:+.5f}")
    owt_tab.to_csv(f"{OUT}/owen_temporal_node{TARGET_NODE}_win{WINDOW_I}.csv", index=False)

    _COLT = {"old": "#9ECAE1", "mid": "#4292C6", "recent": "#08519C"}

    _GAPX, _xof, _x0 = 0.8, {}, 0.0
    for _gi, _g in enumerate(owt_groups):
        if _gi:
            _x0 += _GAPX
        for _t in _g:
            _xof[_t] = _x0
            _x0 += 1.0
    _xpos = [_xof[t] for t in range(_T)]

    fig, ax = plt.subplots(figsize=(7.2, 3.1))
    ax.bar(_xpos, owt_omega, width=.68, color=[_COLT[w] for w in _fam],
           edgecolor="white", linewidth=.6)
    ax.axhline(0, color="#333333", lw=.9)

    _sp = max(abs(v) for v in owt_omega) or 1.0
    for _xv, _v in zip(_xpos, owt_omega):
        ax.text(_xv, _v + (.035 if _v >= 0 else -.035) * _sp, f"{_v:+.3f}",
                ha="center", va="bottom" if _v >= 0 else "top", fontsize=7.2,
                color="#222222")

    for _gi, _g in enumerate(owt_groups[:-1]):
        ax.axvline(_xof[max(_g)] + (1 + _GAPX) / 2, color="#d0d0d0", lw=.7)
    for _w, _g in zip(_GN, owt_groups):
        ax.text((_xof[min(_g)] + _xof[max(_g)]) / 2, 1.02, _w.capitalize(),
                transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=8, fontweight="normal", color="#555555")

    ax.set_xticks(_xpos); ax.set_xticklabels(_tlab, fontsize=7.5)
    ax.tick_params(axis="x", length=0); ax.tick_params(axis="y", labelsize=7.5)
    ax.set_xlabel(f"Historical timestep ({BIN_MIN}-min interval)", fontsize=9)
    ax.set_ylabel(r"Signed Owen value  $\Omega$", fontsize=9)
    ax.set_ylim(-_sp * 1.30, _sp * 1.30)
    ax.grid(axis="y", ls="--", lw=.6, alpha=.25); ax.set_axisbelow(True)
    for _s in ("top", "right"):
        ax.spines[_s].set_visible(False)
    for _s in ("left", "bottom"):
        ax.spines[_s].set_color("#777777"); ax.spines[_s].set_linewidth(.6)

    fig.suptitle("(b) Fine-Grained Temporal Owen Attribution",
                 fontsize=10.5, fontweight="normal", y=.985)

    fig.text(.5, .018,
             r"$\sum_{a \in G_k}\Omega_{i,a}=\Phi^t_{i,G_k}$:    "
             rf"$\Phi^t_{{\mathrm{{Old}}}}$ = {owt_phi_t[0]:+.3f}" + "     "
             rf"$\Phi^t_{{\mathrm{{Mid}}}}$ = {owt_phi_t[1]:+.3f}" + "     "
             rf"$\Phi^t_{{\mathrm{{Recent}}}}$ = {owt_phi_t[2]:+.3f}",
             ha="center", va="top", fontsize=8)

    fig.tight_layout(rect=[0, .055, 1, .955], pad=.4)
    fig.savefig(f"{OUT}/owen_temporal_node{TARGET_NODE}_win{WINDOW_I}.png",
                dpi=600, bbox_inches="tight")
    plt.close(fig)

    _sid = lab["stop_ids"][TARGET_NODE] if lab else TARGET_NODE
    _rid = lab["route_ids"][TARGET_NODE] if lab else "?"

    if _was_tr:
        owt_model.train()
    print("\n  " + "=" * 74)
    print("  READY-TO-PASTE FIGURE CAPTION")
    print("  " + "=" * 74)
    _caption = (
        f"Fine-grained temporal attribution for stop {_sid} on Route {_rid}, test "
        f"window {WINDOW_I}. Exact Owen values under the coalition structure "
        f"{{old, mid, recent}}, with the {_T} historical timesteps t-11, ..., t "
        f"as players; the model predicts the interval t+1. "
        f"EACH TEMPORAL PLAYER REPRESENTS ONE HISTORICAL TIMESTEP ACROSS THE "
        f"TARGET-SPECIFIC SPATIAL CONTEXT comprising the target node, its "
        f"{len(_route)} same-route nodes and {len(_trans)} transfer nodes; "
        f"Omega_t is therefore not the contribution of stop {_sid} alone at that "
        f"timestep. Values are signed and expressed on the model's log1p scale. "
        f"By the Owen group property the timesteps in each window sum exactly to "
        f"that window's reduced-game Shapley value (Phi^t_old = "
        f"{owt_phi_t[0]:+.4f}, Phi^t_mid = {owt_phi_t[1]:+.4f}, Phi^t_recent = "
        f"{owt_phi_t[2]:+.4f}), and all {_T} sum to f - v(empty) = "
        f"{sum(owt_omega):+.4f}, the same total returned by the spatial game and "
        f"by the nine-player spatio-temporal game. Phi^t_recent is the Shapley "
        f"value of the three-player TEMPORAL reduced game and is not the sum of "
        f"the nine-player values phi_self,recent + phi_route,recent + "
        f"phi_transfer,recent, which belong to a different game.")
    import textwrap as _twp
    print(_twp.fill(" ".join(_caption.split()), width=88,
                    initial_indent="    ", subsequent_indent="    "))
    print()
    print(f"\n  figure -> {OUT}/owen_temporal_node{TARGET_NODE}_win{WINDOW_I}.png")
    print(f"  data   -> {OUT}/owen_temporal_node{TARGET_NODE}_win{WINDOW_I}.csv")


## 13. Group Shapley Values


In [ ]:
if True:
    import os, time, math, random, itertools, gc
    import numpy as np
    import pandas as pd
    import torch

    BASE_DIR      = "/speed-scratch/ar_ul/ShapleyGNN"
    CKPT          = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
    ARM_TAG       = "A_guided"          
    OUT_DIR       = f"{BASE_DIR}/results/FIXED_v6_no4683/group_attribution"
    SAMPLE_STRIDE = 13
    NODES_PER_S   = None
    EVAL_BATCH    = 128
    SEED          = 0

    _need = ["load_checkpoint", "get_player_specs", "get_valid_player_mask",
             "apply_player_mask", "resolve_template", "PLAYER_NAMES",
             "SPATIAL_GROUP_PLAYERS", "TEMPORAL_GROUP_PLAYERS", "TA_META"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_missing))
    print("  pre-flight OK")

    model, slots, _ = load_checkpoint(CKPT)
    raw    = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = raw["test_data"].float()
    model.eval()
    device = next(model.parameters()).device

    ve = int(TA_META.get("valid_end", 0) or 0)
    if ve <= 0:
        raise RuntimeError("TA_META['valid_end'] empty - run the cell starting "
                           "'# ---- verify the node/timestep scores are actually "
                           "tie-free ----' first. Expect valid_end = 1773.")
    test_starts = torch.arange(ve, ve + test_x.size(0))
    print(f"  test_x {tuple(test_x.shape)} | device {device} | valid_end {ve}")

    PN     = list(PLAYER_NAMES)
    PARTS  = {"spatial":  [list(g) for g in SPATIAL_GROUP_PLAYERS],
              "temporal": [list(g) for g in TEMPORAL_GROUP_PLAYERS]}
    GNAMES = {"spatial": ["self", "route", "transfer"],
              "temporal": ["old", "mid", "recent"]}

    def _exact_group_shapley(vals, valid_g):
        G, out = len(valid_g), {}
        for gi in valid_g:
            others = [g for g in valid_g if g != gi]
            tot = 0.0
            for r in range(len(others) + 1):
                for S in itertools.combinations(others, r):
                    S = frozenset(S)
                    w = (math.factorial(len(S)) * math.factorial(G - len(S) - 1)
                         / math.factorial(G))
                    tot += w * (vals[S | {gi}] - vals[S])
            out[gi] = tot
        return out

    rng_master = random.Random(SEED)
    sample_ids = list(range(0, test_x.size(0), SAMPLE_STRIDE))
    rec, eff_err = [], []
    t_start = time.time()

    for si in sample_ids:
        x_row = test_x[si:si + 1].to(device)
        tmpl  = resolve_template(slots, int(test_starts[si])).to(device)
        nodes = (list(range(model.N)) if NODES_PER_S is None
                 else rng_master.sample(range(model.N), min(NODES_PER_S, model.N)))

        graphs, keys, plans = [], [], []
        for n in nodes:
            n = int(n)
            specs = get_player_specs(model, n)
            vmask = get_valid_player_mask(specs)
            valid_players = {PN[i] for i in range(len(PN)) if float(vmask[i]) > 0}
            if not valid_players:
                continue
            for mode in ("spatial", "temporal"):
                members = [sorted(set(g) & valid_players) for g in PARTS[mode]]
                valid_g = [g for g in range(3) if members[g]]
                if not valid_g:
                    continue
                subsets = [frozenset(c) for r in range(len(valid_g) + 1)
                           for c in itertools.combinations(valid_g, r)]
                for S in subsets:
                    keep = set().union(*[set(members[g]) for g in S]) if S else set()
                    graphs.append(apply_player_mask(x_row, tmpl, keep, specs))
                    keys.append((n, mode, S))
                plans.append((n, mode, valid_g, members))

        vals_all = {}
        with torch.no_grad():
            for s in range(0, len(graphs), EVAL_BATCH):
                xb = torch.cat(graphs[s:s + EVAL_BATCH], dim=0)
                yb = model(xb)
                for j in range(xb.size(0)):
                    n_j, mode_j, S_j = keys[s + j]
                    vals_all.setdefault((n_j, mode_j), {})[S_j] = \
                        float(yb[j, 0, n_j].item())
                del xb, yb
        model.last_H_seq = None; model.last_branch_outs = None
        del graphs, keys

        for n, mode, valid_g, members in plans:
            vals = vals_all[(n, mode)]
            f_full = vals[frozenset(valid_g)]
            f_ref  = vals[frozenset()]
            phi    = _exact_group_shapley(vals, valid_g)
            eff_err.append(abs(sum(phi.values()) - (f_full - f_ref)))
            tot_abs = sum(abs(v) for v in phi.values()) + 1e-12
            row = dict(arm=ARM_TAG, mode=mode, sample=si, node=n,
                       U=len(valid_g), f_full=f_full, f_ref=f_ref,
                       total=f_full - f_ref,
                       has_transfer=bool(float(model.transfer_valid_mask[n]) > 0))
            for g, gname in enumerate(GNAMES[mode]):
                row[f"phi_{gname}"]   = phi.get(g, np.nan)
                row[f"share_{gname}"] = (abs(phi[g]) / tot_abs) if g in phi else np.nan
            rec.append(row)
        del vals_all, plans
        print(f"  ... window {si}: {len(rec):,} rows "
              f"({time.time()-t_start:.0f}s)", flush=True)
        gc.collect()

    df = pd.DataFrame(rec)
    os.makedirs(OUT_DIR, exist_ok=True)
    csv = f"{OUT_DIR}/group_attribution_{ARM_TAG}.csv"
    df.to_csv(csv, index=False)

    print("\n" + "=" * 78)
    print(f"  rows {len(df):,} | pairs {df.groupby(['sample','node']).ngroups:,} "
          f"| {time.time()-t_start:.0f}s")
    print(f"  EFFICIENCY  max |sum phi - (f_full - f_ref)| = {max(eff_err):.3e}")
    print(f"  saved -> {csv}")
    print("=" * 78)

    for mode in ("spatial", "temporal"):
        for U in sorted(df[df["mode"] == mode]["U"].unique()):
            sub = df[(df["mode"] == mode) & (df["U"] == U)]
            if sub.empty:
                continue
            print(f"\n  {mode.upper()}  U={U}  -  exact group Shapley, arm {ARM_TAG}"
                  f"  ({len(sub):,} pairs)")
            print(f"  {'group':>10} {'n pairs':>9} {'mean phi':>11} {'mean |phi|':>11} "
                  f"{'share |phi|':>12} {'phi > 0':>9}")
            print("  " + "-" * 68)
            for gname in GNAMES[mode]:
                c = sub[f"phi_{gname}"].dropna()
                if c.empty:
                    continue
                print(f"  {gname:>10} {len(c):>9,} {c.mean():>+11.4f} "
                      f"{c.abs().mean():>11.4f} "
                      f"{sub[f'share_{gname}'].dropna().mean():>11.1%} "
                      f"{(c > 0).mean():>8.0%}")
            print(f"  {'TOTAL':>10} {len(sub):>9,} {sub['total'].mean():>+11.4f}")
    print("\n  mean phi = signed average (direction). mean |phi| = magnitude.")
    print("  share |phi| = that group's fraction of the total attribution magnitude.")
    print("  'phi > 0' = how often the group PUSHES THE PREDICTION UP; a group near")
    print("  0% consistently pulls it down, which is what group OPPOSITION looks like.")


## 14. Global Feature Importance


### 14.1 Instance-Level Attribution


In [ ]:
if True:
    import os, time
    import numpy as np, pandas as pd, torch

    BASE_DIR = "/speed-scratch/ar_ul/ShapleyGNN"
    ARM_SEED = 1
    SEED     = 0
    N_PAIRS  = 1500
    CKPT     = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed{ARM_SEED}.pt"
    OUT      = f"{BASE_DIR}/results/FIXED_v6_no4683/global_importance"

    _need = ["load_checkpoint", "resolve_template", "reference_branch_outs_target",
             "estimate_group_shapley", "PLAYER_NAMES", "TA_META", "IDX_TRANSFER"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_missing))
    if not os.path.isfile(CKPT):
        raise FileNotFoundError(CKPT)
    os.makedirs(OUT, exist_ok=True)
    if os.path.exists(f"{OUT}/phi_per_pair_seed{ARM_SEED}.csv"):
        raise RuntimeError(f"phi_per_pair_seed{ARM_SEED}.csv already exists")
    print("  pre-flight OK")

    model, slots, _ = load_checkpoint(CKPT)
    _raw   = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = _raw["test_data"].float()
    n_test = test_x.shape[0]
    del _raw["state_dict"]
    dev = next(model.parameters()).device
    _was_training = model.training
    model.eval()
    if model.N != test_x.shape[-1]:
        raise RuntimeError(f"node mismatch: model.N={model.N} vs {test_x.shape[-1]}")

    _st = _raw.get("test_starts")
    if _st is not None:
        test_starts = np.asarray(_st.cpu() if hasattr(_st, "cpu") else _st, dtype=np.int64)
        if len(test_starts) != n_test:
            raise RuntimeError("stored test_starts length mismatch")
        _ve = int(test_starts[0])
    else:
        _ve = int(TA_META.get("valid_end", 0) or 0) or int(np.floor((1983 - 12) * 0.9))
        if int(np.floor(0.9 * (_ve + n_test))) != _ve:
            raise RuntimeError(f"valid_end {_ve} inconsistent with {n_test} test windows")
        test_starts = _ve + np.arange(n_test, dtype=np.int64)
    print(f"  seed {ARM_SEED} | windows {n_test} | stops {model.N} | valid_end {_ve}")

    transfer_ok = (model.transfer_valid_mask > 0).cpu().numpy()

    rng   = np.random.default_rng(SEED)
    flat  = rng.choice(n_test * model.N, size=N_PAIRS, replace=False)
    pairs = [(int(k // model.N), int(k % model.N)) for k in flat]
    print(f"  sampled {N_PAIRS:,} DISTINCT pairs from {n_test*model.N:,}")

    @torch.no_grad()
    def _ctilde(i, n):
        tmpl = resolve_template(slots, int(test_starts[i])).to(dev)
        c, _, _ = reference_branch_outs_target(model, test_x[i:i+1].to(dev), tmpl, n)
        model.last_H_seq = None; model.last_branch_outs = None
        return c.detach().cpu().numpy().astype(np.float64)

    @torch.no_grad()
    def _phi(i, n):
        tmpl = resolve_template(slots, int(test_starts[i])).to(dev)
        d = estimate_group_shapley(model=model, inputs=test_x[i:i+1].to(dev),
                                   target_node=n, baseline_template=tmpl, exact=True)
        model.last_H_seq = None; model.last_branch_outs = None
        return np.array([float(d.get(p, 0.0)) for p in PLAYER_NAMES], dtype=np.float64)

    C = np.zeros((N_PAIRS, 9)); P = np.zeros((N_PAIRS, 9))
    for tag, fn, M, every in (("c~", _ctilde, C, 500), ("phi", _phi, P, 100)):
        t0 = time.time()
        for k, (i, n) in enumerate(pairs):
            M[k] = fn(i, n)
            if (k + 1) % every == 0:
                print(f"    {tag} {k+1}/{N_PAIRS}  ({(time.time()-t0)/60:.1f} min)", flush=True)
        print(f"  {tag} done in {(time.time()-t0)/60:.1f} min")

    if not (np.all(np.isfinite(P)) and np.all(np.isfinite(C))):
        raise RuntimeError("non-finite attributions")
    stop_idx = np.array([n for _, n in pairs]); has_tr = transfer_ok[stop_idx]
    if (~has_tr).any():
        bad = float(np.abs(P[~has_tr][:, IDX_TRANSFER]).max())
        print(f"  max |phi| on absent transfer players: {bad:.3e}")
        if bad > 1e-7:
            raise RuntimeError("player bookkeeping broken")

    for M, nm in ((C, "ctilde"), (P, "phi")):
        pd.DataFrame(M, columns=PLAYER_NAMES).assign(
            window=[i for i, _ in pairs],
            absolute_start=[int(test_starts[i]) for i, _ in pairs],
            stop=stop_idx, has_transfer=has_tr
        ).to_csv(f"{OUT}/{nm}_per_pair_seed{ARM_SEED}.csv", index=False)

    a = np.abs(C).mean(0); b = np.abs(P).mean(0)
    ra, rb = 100 * a / a.sum(), 100 * b / b.sum()
    print(f"\n  {'player':<17}{'R_p c~ %':>10}{'R_p phi %':>11}{'gap pp':>9}")
    for j in np.argsort(-rb):
        print(f"  {PLAYER_NAMES[j]:<17}{ra[j]:>10.1f}{rb[j]:>11.1f}{ra[j]-rb[j]:>+9.1f}")
    print(f"  max gap {np.abs(ra-rb).max():.1f} pp   TVD {0.5*np.abs(ra-rb).sum():.1f} pp")

    if _was_training:
        model.train()
    print(f"\n  saved -> {OUT}/ctilde_per_pair_seed{ARM_SEED}.csv, "
          f"phi_per_pair_seed{ARM_SEED}.csv")


### 14.2 Global Importance Across Seeds


In [ ]:
if True:
    import numpy as np, pandas as pd
    import matplotlib.pyplot as plt

    OUT   = "/speed-scratch/ar_ul/ShapleyGNN/results/FIXED_v6_no4683/global_importance"
    SEEDS = [0, 1, 3]
    META  = ["window", "absolute_start", "stop", "has_transfer"]

    Ra, Rb, key0 = [], [], None
    for s in SEEDS:
        C = pd.read_csv(f"{OUT}/ctilde_per_pair_seed{s}.csv")
        P = pd.read_csv(f"{OUT}/phi_per_pair_seed{s}.csv")
        pl  = [c for c in P.columns if c not in META]
        key = tuple(map(tuple, (P.window, P.absolute_start, P.stop)))
        assert len(pl) == 9 and list(C.columns) == list(P.columns), f"seed {s}: bad columns"
        assert tuple(map(tuple, (C.window, C.absolute_start, C.stop))) == key, \
            f"seed {s}: c~/phi misaligned"
        if key0 is None:
            key0, PLAYERS = key, pl
        assert key == key0, f"seed {s}: different pairs -- was the generating SEED changed?"
        for M, R in ((C, Ra), (P, Rb)):
            v = np.abs(M[PLAYERS].to_numpy(float)).mean(0)
            R.append(100 * v / v.sum())
        print(f"  seed {s}: {len(P):,} pairs")

    n = len(Ra)
    A, B = np.vstack(Ra), np.vstack(Rb)
    am, asd, bm, bsd = A.mean(0), A.std(0, ddof=1), B.mean(0), B.std(0, ddof=1)
    gap, tvd = np.abs(A - B).max(1), 0.5 * np.abs(A - B).sum(1)
    top = A.argmax(1) == B.argmax(1)

    o = np.argsort(-bm)
    print(f"\n  n = {n} seeds {SEEDS}")
    print(f"  {'player':<17}{'R_p c~ %':>18}{'R_p phi %':>18}{'Delta pp':>10}")
    for j in o:
        print(f"  {PLAYERS[j]:<17}{am[j]:>11.1f} +/-{asd[j]:>5.1f}"
              f"{bm[j]:>11.1f} +/-{bsd[j]:>5.1f}{am[j]-bm[j]:>+10.1f}")
    print(f"\n  max gap          {gap.mean():.1f} +/- {gap.std(ddof=1):.1f} pp   {np.round(gap,1)}")
    print(f"  total variation  {tvd.mean():.1f} +/- {tvd.std(ddof=1):.1f} pp   {np.round(tvd,1)}")
    print(f"  same top player  {top.sum()}/{n} seeds   {top}")

    o = o[::-1]; y = np.arange(9)
    fig, ax = plt.subplots(figsize=(8.4, 5.5))
    for dy, m, col, lab in ((+.20, bm, "#6FA8DC", r"exact Shapley attribution"),
                            (-.20, am, "#F4A261", r"intrinsic contribution")):
        ax.barh(y + dy, m[o], height=.38, color=col, edgecolor="white", label=lab)
        for k in range(9):
            ax.text(m[o][k] + .6, y[k] + dy, f"{m[o][k]:.1f}",
                    va="center", fontsize=8.2, color="#333333")
    ax.set_yticks(y); ax.set_yticklabels([PLAYERS[j] for j in o], fontsize=10)
    ax.set_xlabel(r"$R_p$ = relative global importance   (%)", fontsize=10.5)
    ax.set_title("Global player importance — arm A (guided), all stops\n"
                 f"mean over {n} seeds · {len(key0[0]):,} pairs per seed · "
                 f"max gap {gap.mean():.1f} pp, "
                 f"total variation {tvd.mean():.1f} pp",
                 fontsize=11.5, fontweight="bold")
    ax.legend(fontsize=9.5, loc="lower right", framealpha=0.92)
    ax.grid(axis="x", ls="--", alpha=.35); ax.set_axisbelow(True)
    ax.set_xlim(0, float(max(am.max(), bm.max())) * 1.16)
    fig.tight_layout()
    png = f"{OUT}/global_shares_ctilde_vs_phi_all_mean{n}.png"
    fig.savefig(png, dpi=300, bbox_inches="tight"); plt.close(fig)
    print(f"\n  figure -> {png}")


## 15. Group-Level Fidelity and Sparsity


In [ ]:
if True:
    import os, time, math, random, itertools, gc
    import numpy as np
    import pandas as pd
    import torch

    BASE_DIR      = "/speed-scratch/ar_ul/ShapleyGNN"
    CKPT          = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
    ARM_TAG       = "A_guided"          
    OUT_DIR       = f"{BASE_DIR}/results/FIXED_v6_no4683/group_fidelity"
    SAMPLE_STRIDE = 13
    NODES_PER_S   = None
    SEED          = 0

    _need = ["load_checkpoint", "get_player_specs", "get_valid_player_mask",
             "apply_player_mask", "resolve_template", "reference_branch_outs_target",
             "PLAYER_NAMES", "SPATIAL_GROUP_PLAYERS", "TEMPORAL_GROUP_PLAYERS",
             "TA_META"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - missing: " + ", ".join(_missing))
    print("  pre-flight OK")

    model, slots, _ = load_checkpoint(CKPT)
    raw    = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = raw["test_data"].float()
    model.eval()
    device = next(model.parameters()).device

    ve = int(TA_META.get("valid_end", 0) or 0)
    if ve <= 0:
        raise RuntimeError(
            "TA_META['valid_end'] is empty. Fidelity uses the TIME-AWARE baseline, "
            "so real window starts are required (unlike the TA-11 timing cell).\n"
            "Run the cell starting '# ---- verify the node/timestep scores are "
            "actually tie-free ----' first, then re-run this. Expect valid_end = 1773.")
    test_starts = torch.arange(ve, ve + test_x.size(0))
    print(f"  test_x {tuple(test_x.shape)} | device {device} | valid_end {ve}")

    PN      = list(PLAYER_NAMES)
    PARTS   = {"spatial":  [list(g) for g in SPATIAL_GROUP_PLAYERS],
               "temporal": [list(g) for g in TEMPORAL_GROUP_PLAYERS]}
    GNAMES  = {"spatial": ["self", "route", "transfer"],
               "temporal": ["old", "mid", "recent"]}

    def _exact_group_shapley(vals, valid_g):
        G = len(valid_g)
        out = {}
        for gi in valid_g:
            others = [g for g in valid_g if g != gi]
            tot = 0.0
            for r in range(len(others) + 1):
                for S in itertools.combinations(others, r):
                    S = frozenset(S)
                    w = (math.factorial(len(S)) * math.factorial(G - len(S) - 1)
                         / math.factorial(G))
                    tot += w * (vals[S | {gi}] - vals[S])
            out[gi] = tot
        return out

    rng_master = random.Random(SEED)
    sample_ids = list(range(0, test_x.size(0), SAMPLE_STRIDE))
    rec, eff_err = [], []
    t_start = time.time()

    for si in sample_ids:
        x_row = test_x[si:si + 1].to(device)
        tmpl  = resolve_template(slots, int(test_starts[si])).to(device)
        with torch.no_grad():
            y_raw = model(x_row)
        model.last_H_seq = None; model.last_branch_outs = None
        nodes = (range(model.N) if NODES_PER_S is None
                 else rng_master.sample(range(model.N), min(NODES_PER_S, model.N)))

        for n in nodes:
            n     = int(n)
            specs = get_player_specs(model, n)
            vmask = get_valid_player_mask(specs)
            valid_players = {PN[i] for i in range(len(PN)) if float(vmask[i]) > 0}
            if not valid_players:
                continue

            c_vec = reference_branch_outs_target(model, x_row, tmpl, n)[0]
            c_vec = c_vec.detach().cpu().numpy()

            for mode in ("spatial", "temporal"):
                members = [sorted(set(g) & valid_players) for g in PARTS[mode]]
                valid_g = [g for g in range(3) if members[g]]
                U = len(valid_g)
                if U == 0:
                    continue

                subsets = [frozenset(c) for r in range(U + 1)
                           for c in itertools.combinations(valid_g, r)]
                keeps   = [set().union(*[set(members[g]) for g in S]) if S else set()
                           for S in subsets]
                with torch.no_grad():
                    xb = torch.cat([apply_player_mask(x_row, tmpl, k, specs)
                                    for k in keeps], dim=0)
                    yb = model(xb)
                    vals = {S: float(yb[i, 0, n].item()) for i, S in enumerate(subsets)}
                    del xb, yb
                model.last_H_seq = None; model.last_branch_outs = None

                full, empty = frozenset(valid_g), frozenset()
                f_full, f_ref = vals[full], vals[empty]
                f_raw = float(y_raw[0, 0, n].item())
                den_eq25 = abs(f_raw  - f_ref)
                den_game = abs(f_full - f_ref)

                phi = _exact_group_shapley(vals, valid_g)
                eff_err.append(abs(sum(phi.values()) - (f_full - f_ref)))

                ct = {g: float(np.abs([c_vec[PN.index(p)] for p in members[g]]).sum())
                      for g in valid_g}

                plans = []
                for basis, score in (("phi", {g: abs(phi[g]) for g in valid_g}),
                                     ("ctilde", ct)):
                    od = sorted(valid_g, key=lambda g: -score[g])
                    plans.append((basis, "model",   0, od))
                    plans.append((basis, "reverse", 0, od[::-1]))

                for basis, ordering, d, order in plans:
                    for m in range(U + 1):
                        kept = frozenset(order[m:])
                        fm   = vals[kept]
                        rec.append(dict(
                            arm=ARM_TAG, mode=mode, basis=basis,
                            ordering=ordering, draw=d, sample=si, node=n,
                            U=U, m=m, sparsity=1.0 - m / U,
                            fid=abs(f_raw - fm),
                            fid_signed=(f_raw - fm),
                            fid_norm=abs(f_raw - fm) / (den_eq25 + 1e-12),
                            fid_game=abs(f_full - fm),
                            fid_game_norm=abs(f_full - fm) / (den_game + 1e-12),
                            f_raw=f_raw, f_full=f_full, f_ref=f_ref,
                            has_transfer=bool(float(model.transfer_valid_mask[n]) > 0)))
        print(f"  ... window {si}: {len(rec):,} rows "
              f"({time.time()-t_start:.0f}s)", flush=True)
        gc.collect()

    df = pd.DataFrame(rec)
    os.makedirs(OUT_DIR, exist_ok=True)
    csv = f"{OUT_DIR}/group_fidelity_exact_{ARM_TAG}.csv"
    df.to_csv(csv, index=False)

    print("\n" + "=" * 74)
    print(f"  rows {len(df):,} | pairs {df.groupby(['sample','node']).ngroups:,} "
          f"| {time.time()-t_start:.0f}s")
    print(f"  EFFICIENCY CHECK  max |sum phi - (f_full - f_empty)| = {max(eff_err):.3e}"
          "   (exact enumeration -> should be ~1e-6 or less)")
    _bg = (df["f_raw"] - df["f_full"]).abs().mean()
    _sp = (df["f_raw"] - df["f_ref"]).abs().mean()
    print(f"  BACKGROUND SHARE  mean|f_raw - f_full| / mean|f_raw - f_ref| = "
          f"{_bg/(_sp+1e-12):.1%}")
    print( "    (how much of the prediction lives OUTSIDE the nine players.")
    print( "     'fid' uses f_raw and matches your eq.25 code; 'fid_game' uses")
    print( "     f_full and is the self-contained Shapley game. Report one.)")
    print(f"  saved -> {csv}")
    print("=" * 74)

    for mode in ("spatial", "temporal"):
        for U in sorted(df[df["mode"] == mode]["U"].unique()):
            sub = df[(df["mode"] == mode) & (df["U"] == U)]
            npair = sub.groupby(["sample", "node"]).ngroups
            print(f"\n  {mode.upper()}  U={U} units  ({npair:,} pairs)")
            _den = (sub["f_raw"] - sub["f_ref"]).abs().mean()
            piv = (sub.groupby(["basis", "ordering", "m"])
                      .agg(S=("sparsity", "mean"),
                           F=("fid", "mean"),
                           SE=("fid", lambda v: v.std(ddof=1) / max(1, len(v))**0.5))
                      .reset_index())
            piv["Fn"] = piv["F"] / (_den + 1e-12)
            print("      first number = eq.25 Fidelity (report this); "
                  "(brackets) = normalised, reading aid only")
            for basis in ("phi", "ctilde"):
                print(f"    basis = {basis}")
                print(f"      {'m':>2} {'S+':>6} | " +
                      " | ".join(f"{o:>20}" for o in ("model (Fid+)", "reverse (Fid-)")))
                for m in range(U + 1):
                    cells = []
                    for o in ("model", "reverse"):
                        r = piv[(piv.basis == basis) & (piv.ordering == o) & (piv.m == m)]
                        cells.append(f"{r.F.iloc[0]:.4f}+/-{r.SE.iloc[0]:.4f}"
                                     f"({r.Fn.iloc[0]:.2f})"
                                     if len(r) else " " * 20)
                    _sr = piv[piv.m == m]
                    S = _sr.S.iloc[0] if len(_sr) else float("nan")
                    print(f"      {m:>2} {S:6.3f} | " + " | ".join(f"{c:>20}" for c in cells))

    PLOT_BASIS = "phi"          
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        series = [("spatial",  3, "tab:blue",   "o", "Spatial, interchange stops (U=3)"),
                  ("spatial",  2, "tab:cyan",   "s", "Spatial, single-route stops (U=2)"),
                  ("temporal", 3, "tab:red",    "^", "Temporal (U=3)")]

        fig, ax = plt.subplots(figsize=(6.4, 4.4))
        for mode, U, col, mk, lab in series:
            g = (df[(df["mode"] == mode) & (df["U"] == U)
                    & (df["basis"] == PLOT_BASIS) & (df["ordering"] == "model")]
                 .groupby("sparsity")["fid"].agg(["mean", "sem"])
                 .reset_index().sort_values("sparsity"))
            if g.empty:
                continue
            ax.errorbar(g["sparsity"], g["mean"], yerr=g["sem"], color=col,
                        marker=mk, markersize=5, linewidth=1.7, capsize=3, label=lab)

        ax.set_xlabel("Sparsity$^+$   $(1 - |m_i| / |M_i|)$")
        ax.set_ylabel(r"Fidelity$^+$   $\frac{1}{Q}\sum_i |f(\mathcal{G}_i) - f(\mathcal{G}_i^{1-m_i})|$")
        ax.set_title(f"Group-level Fidelity$^+$ vs Sparsity$^+$ - exact groupShapley - {ARM_TAG}")
        ax.invert_xaxis()
        ax.grid(alpha=.3)
        ax.legend(fontsize=8, loc="best", frameon=True)
        fig.tight_layout()
        png = f"{OUT_DIR}/group_fidelity_vs_sparsity_{ARM_TAG}.png"
        fig.savefig(png, dpi=200); plt.close(fig)
        print(f"\n  plot -> {png}")
        print(f"  (Fidelity+ only, basis={PLOT_BASIS}, unnormalised eq.25 values,")
        print( "   x reversed: Sparsity+ 1.0 at left -> 0.0 at right, curves rise)")
    except Exception as e:
        print(f"  (plotting skipped: {e})")


## 16. Fine-Grained Fidelity and Sparsity


### 16.1 Node- and Timestep-Level Fidelity


In [ ]:
if True:
    import os, time, random, gc
    import numpy as np
    import pandas as pd
    import torch

    PILOT          = False

    BASE_DIR       = "/speed-scratch/ar_ul/ShapleyGNN"
    CKPT           = f"{BASE_DIR}/checkpoints/FIXED_v6_no4683/ablation_A_guided_seed0.pt"
    ARM_TAG        = "A_guided"       
    OUT_DIR        = f"{BASE_DIR}/results/FIXED_v6_no4683/node_fidelity"
    SAMPLE_STRIDE  = 13
    STOPS_PER_WIN  = 40
    N_PERM         = 200
    SPARSITY_GRID  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    N_PILOT        = 4
    EXACT_TEMPORAL = True
    CHECK_GROUPS   = True
    MASK_MODE      = "timeaware"
    SEED           = 0

    _need = ["load_checkpoint", "estimate_owen_values", "exact_owen_values",
             "exact_quotient_shapley", "apply_unit_mask", "_involved_sets",
             "order_units", "node_level_scores", "reference_branch_outs_target",
             "resolve_template", "TA_META", "WINDOWS",
             "IDX_SELF", "IDX_ROUTE", "IDX_TRANSFER",
             "IDX_OLD", "IDX_MID", "IDX_RECENT"]
    _missing = [f for f in _need if f not in globals()]
    if _missing:
        raise RuntimeError("PRE-FLIGHT FAILED - run PART 1 (the cell above) and the "
                           "core cells first.\n  missing: " + ", ".join(_missing))
    print("  pre-flight OK")

    model, slots, _ = load_checkpoint(CKPT)
    raw    = torch.load(CKPT, map_location="cpu", weights_only=False)
    test_x = raw["test_data"].float()
    model.eval()
    device = next(model.parameters()).device

    ve = int(TA_META.get("valid_end", 0) or 0)
    if ve <= 0:
        raise RuntimeError("TA_META['valid_end'] empty - run the cell starting "
                           "'# ---- verify the node/timestep scores are actually "
                           "tie-free ----' first. Expect valid_end = 1773.")
    test_starts = torch.arange(ve, ve + test_x.size(0))
    print(f"  test_x {tuple(test_x.shape)} | device {device} | valid_end {ve}")
    print(f"  sparsity grid (fraction masked): {SPARSITY_GRID}")

    FRACS = np.asarray(SPARSITY_GRID, dtype=float)

    def _sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    if PILOT:
        A_sp = np.array([sum(len(s) for s in _involved_sets(model, n))
                         for n in range(model.N)])
        print(f"\n  |V(n)| spatial : min {A_sp.min()}  median {int(np.median(A_sp))}  "
              f"max {A_sp.max()}  mean {A_sp.mean():.1f}")
        print( "  temporal atoms : 12 (fixed)")
        rng   = random.Random(SEED)
        pairs = [(rng.randrange(test_x.size(0)), rng.randrange(model.N))
                 for _ in range(N_PILOT + 1)]
        si, n = pairs[0]
        estimate_owen_values(model, test_x[si:si+1].to(device),
                             resolve_template(slots, int(test_starts[si])).to(device),
                             n, "spatial", n_perm=8, seed=0)
        _sync()
        per = {"sp_sampled": [], "tp_sampled": [], "tp_exact": []}
        for si, n in pairs[1:]:
            x    = test_x[si:si+1].to(device)
            tmpl = resolve_template(slots, int(test_starts[si])).to(device)
            for key, mode in (("sp_sampled", "spatial"), ("tp_sampled", "temporal")):
                _sync(); t0 = time.time()
                estimate_owen_values(model, x, tmpl, n, mode, n_perm=N_PERM,
                                     mask_mode=MASK_MODE, seed=SEED)
                _sync(); per[key].append(time.time() - t0)
            _sync(); t0 = time.time()
            _o, _g, _mt = exact_owen_values(model, x, tmpl, n, "temporal",
                                            mask_mode=MASK_MODE)
            _sync(); per["tp_exact"].append(time.time() - t0)
            print(f"    stop {n:>4} |V|={A_sp[n]:>3} :  spatial {per['sp_sampled'][-1]:6.2f}s"
                  f"   temporal sampled {per['tp_sampled'][-1]:6.2f}s"
                  f"   temporal exact {per['tp_exact'][-1]:6.2f}s"
                  f" ({_mt['n_coalitions']} coalitions)")
        s_sp = float(np.median(per["sp_sampled"]))
        s_tp = float(np.median(per["tp_sampled"]))
        s_te = float(np.median(per["tp_exact"]))
        drawn    = np.array([A_sp[n] for _, n in pairs[1:]], dtype=float)
        per_pair = (s_sp * (A_sp.mean() / max(drawn.mean(), 1e-9))
                    + (s_te + s_tp if EXACT_TEMPORAL else s_tp))
        print(f"\n  median: spatial {s_sp:.2f}s | temporal sampled {s_tp:.2f}s "
              f"| temporal exact {s_te:.2f}s")
        print(f"  exact temporal is {s_tp / max(s_te, 1e-9):.1f}x "
              f"{'cheaper' if s_te < s_tp else 'MORE EXPENSIVE'} than sampled")
        print(f"  per (window, stop), everything the full run does: {per_pair:.2f}s")
        print( "  NOTE: spatial is re-weighted assuming cost is linear in |V(n)|.")
        print( "  Batching makes it sub-linear, so this is an UPPER bound.")
        print("\n  " + "=" * 60)
        print(f"  {'windows':>8} {'stops/win':>10} {'pairs':>8} {'estimate':>16}")
        print("  " + "-" * 60)
        for W, S in ((16, 515), (16, 200), (16, 80), (16, 40), (8, 40)):
            sec = W * S * per_pair; hh = sec / 3600
            print(f"  {W:>8} {S:>10} {W*S:>8,} "
                  f"{(f'{sec/60:.0f} min' if hh < 1 else f'{hh:.1f} h'):>16}")
        print("  " + "=" * 60)
        print("  Sussek & Heindorf average 200 explanations per sparsity level;")
        print("  16 x 40 = 640 pairs already exceeds that.")
        print("\n  PILOT ONLY - set PILOT = False and STOPS_PER_WIN, then re-run.")

    else:
        rng_master = random.Random(SEED)
        sample_ids = list(range(0, test_x.size(0), SAMPLE_STRIDE))
        print(f"  {len(sample_ids)} windows x {STOPS_PER_WIN} stops "
              f"= {len(sample_ids) * STOPS_PER_WIN:,} pairs")
        rec, eff_err, grp_err = [], [], []
        t_start = time.time()

        for si in sample_ids:
            x_row = test_x[si:si + 1].to(device)
            tmpl  = resolve_template(slots, int(test_starts[si])).to(device)
            nodes = rng_master.sample(range(model.N), min(STOPS_PER_WIN, model.N))
            for n in nodes:
                n = int(n)
                self_n, route_n, trans_n = _involved_sets(model, n)
                involved = self_n + route_n + trans_n
                if len(involved) < 2:
                    continue

                with torch.no_grad():
                    node_w, time_w, _ = node_level_scores(model, x_row, tmpl, n)
                node_w = node_w.detach().cpu().numpy()
                time_w = time_w.detach().cpu().numpy()
                c_vec  = reference_branch_outs_target(model, x_row, tmpl, n)[0]
                c_vec  = c_vec.detach().cpu().numpy()
                cg = {"nodes": [float(np.abs(c_vec[IDX_SELF]).sum()),
                                float(np.abs(c_vec[IDX_ROUTE]).sum()),
                                float(np.abs(c_vec[IDX_TRANSFER]).sum())],
                      "times": [float(np.abs(c_vec[IDX_OLD]).sum()),
                                float(np.abs(c_vec[IDX_MID]).sum()),
                                float(np.abs(c_vec[IDX_RECENT]).sum())]}
                gid = {"nodes": {int(n): 0}, "times": {}}
                gid["nodes"].update({int(j): 1 for j in route_n})
                gid["nodes"].update({int(j): 2 for j in trans_n})
                for g, w in enumerate(("old", "mid", "recent")):
                    gid["times"].update({int(t): g for t in WINDOWS[w]})

                exact_grp = {}
                runs = [("spatial", "sampled")]
                runs += ([("temporal", "exact"), ("temporal", "sampled")]
                         if EXACT_TEMPORAL else [("temporal", "sampled")])

                for mode, est in runs:
                    if est == "exact":
                        owen, grp, meta = exact_owen_values(
                            model, x_row, tmpl, n, mode, mask_mode=MASK_MODE)
                    else:
                        owen, grp, meta = estimate_owen_values(
                            model, x_row, tmpl, n, mode, n_perm=N_PERM,
                            mask_mode=MASK_MODE, seed=rng_master.randrange(10 ** 9))
                    if not owen:
                        continue
                    atoms, kind = meta["atoms"], meta["kind"]
                    U = len(atoms)
                    if U < 2:
                        continue
                    f_raw, f_ref = meta["f_full"], meta["f_empty"]
                    eff_err.append(abs(sum(owen.values()) - (f_raw - f_ref)))

                    if est == "exact":
                        exact_grp[mode] = dict(grp)
                    if est == "sampled" and CHECK_GROUPS:
                        ref = exact_grp.get(mode)
                        if ref is None:
                            ref, _ = exact_quotient_shapley(
                                model, x_row, tmpl, n, mode, mask_mode=MASK_MODE)
                        for gk, ex in ref.items():
                            sm = grp.get(gk, 0.0)
                            grp_err.append(dict(mode=mode, group=gk, sampled=sm,
                                                exact=ex, abs_err=abs(sm - ex),
                                                scale=abs(ex)))

                    scores = {"owen": np.array([abs(owen[a]) for a in atoms],
                                               dtype=float)}
                    if est == "sampled":      
                        scores["deviation"] = np.array(
                            [cg[kind][gid[kind][int(a)]] *
                             (node_w if kind == "nodes" else time_w)[int(a)]
                             for a in atoms], dtype=float)

                    for basis, sc in scores.items():
                        for ordering in ("model", "reverse"):
                            order = order_units(sc, ordering, None)
                            fm_cache = {0: f_raw}
                            for f in FRACS:
                                m = int(round(f * U))
                                if m not in fm_cache:
                                    sel = [[atoms[i]] for i in order[:m]]
                                    xm  = apply_unit_mask(x_row, tmpl, sel, kind,
                                                          involved, MASK_MODE)
                                    with torch.no_grad():
                                        fm_cache[m] = float(
                                            model(xm.to(device))[0, 0, n].item())
                                    model.last_H_seq = None
                                    model.last_branch_outs = None
                                fm = fm_cache[m]
                                rec.append(dict(
                                    arm=ARM_TAG, mode=mode, estimator=est,
                                    basis=basis, ordering=ordering,
                                    sample=si, node=n, U=U, m=m,
                                    frac=round(float(f), 4),
                                    masked_frac=m / U,
                                    sparsity=1.0 - m / U,
                                    fid=abs(f_raw - fm), fid_signed=f_raw - fm,
                                    f_raw=f_raw, f_ref=f_ref,
                                    den=abs(f_raw - f_ref),
                                    has_transfer=bool(trans_n)))
            print(f"  ... window {si}: {len(rec):,} rows "
                  f"({time.time() - t_start:.0f}s)", flush=True)
            gc.collect()

        df = pd.DataFrame(rec)
        os.makedirs(OUT_DIR, exist_ok=True)
        csv = f"{OUT_DIR}/node_fidelity_owen_{ARM_TAG}.csv"
        df.to_csv(csv, index=False)

        print("\n" + "=" * 76)
        print(f"  rows {len(df):,} | pairs {df.groupby(['sample','node']).ngroups:,}"
              f" | {time.time() - t_start:.0f}s")
        print(f"  EFFICIENCY  max |sum owen - (f_full - f_empty)| = {max(eff_err):.3e}")
        print( "    SAMPLED runs: efficiency telescopes within each permutation, so it")
        print( "    holds at ANY n_perm - it validates the MASKING, not the sampling.")
        print( "    EXACT runs: a real correctness check on the eq.(2) weights.")
        print(f"  saved -> {csv}")

        if grp_err:
            ge = pd.DataFrame(grp_err)
            ge.to_csv(f"{OUT_DIR}/owen_group_error_{ARM_TAG}.csv", index=False)
            print("\n  " + "-" * 72)
            print("  OWEN SAMPLING ERROR vs GROUND TRUTH")
            print("    sum_{a in B_k} Ow_a  (sampled)   vs   Sh_k(quotient)  (exact)")
            print("    Equal by Owen's definition. Does NOT telescope, so this is a")
            print(f"    real error measurement at n_perm = {N_PERM}.")
            print(f"    {'mode':>9} {'n':>7} {'mean|err|':>11} {'p95|err|':>11} "
                  f"{'mean rel err':>13}")
            for mode in sorted(ge["mode"].unique()):
                q = ge[ge["mode"] == mode]
                print(f"    {mode:>9} {len(q):>7,} {q['abs_err'].mean():>11.5f} "
                      f"{q['abs_err'].quantile(.95):>11.5f} "
                      f"{(q['abs_err'] / (q['scale'] + 1e-12)).mean():>12.1%}")
            print("  " + "-" * 72)
        print("=" * 76)

        for mode, est in (("spatial", "sampled"), ("temporal", "exact"),
                          ("temporal", "sampled")):
            sub = df[(df["mode"] == mode) & (df["estimator"] == est)]
            if sub.empty:
                continue
            den   = sub["den"].mean()
            npair = sub.groupby(["sample", "node"]).ngroups
            print(f"\n  {mode.upper()} [{est}]  {npair:,} pairs, "
                  f"U {sub['U'].min()}-{sub['U'].max()} (mean {sub['U'].mean():.1f})")
            print( "    Fidelity = eq.25, unnormalised (report this). "
                   "(brackets) = /mean|f-f_ref|, reading aid.")
            piv = (sub.groupby(["basis", "ordering", "frac"])
                      .agg(S=("masked_frac", "mean"), F=("fid", "mean"),
                           SE=("fid", lambda v: v.std(ddof=1) / max(1, len(v)) ** .5))
                      .reset_index())
            piv["Fn"] = piv["F"] / (den + 1e-12)
            for basis in ("owen", "deviation"):
                if basis not in set(piv["basis"]):
                    continue
                print(f"    basis = {basis}")
                print(f"      {'Sparsity':>8} | {'Fidelity+':>22} | {'Fidelity-':>22}")
                for f in sorted(piv["frac"].unique()):
                    cells, S = [], float("nan")
                    for o in ("model", "reverse"):
                        r = piv[(piv.basis == basis) & (piv.ordering == o)
                                & (piv.frac == f)]
                        if len(r):
                            S = r.S.iloc[0]
                            cells.append(f"{r.F.iloc[0]:.4f}+/-{r.SE.iloc[0]:.4f}"
                                         f"({r.Fn.iloc[0]:.2f})")
                        else:
                            cells.append(" " * 22)
                    print(f"      {S:8.3f} | " + " | ".join(f"{c:>22}" for c in cells))

        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            _has_exact = (df["estimator"] == "exact").any()
            for mode, unit in (("spatial", "stops"), ("temporal", "timesteps")):
                fig, ax = plt.subplots(figsize=(6.0, 4.2))
                drew = False
                for basis, ls, col, mk, lab in (
                        ("owen",      "-",  "#1f4e79", "o", "Owen"),
                        ("deviation", "--", "#d95f02", "s", "deviation")):
                    est = ("exact" if (basis == "owen" and mode == "temporal"
                                       and _has_exact) else "sampled")
                    q = df[(df["mode"] == mode) & (df["estimator"] == est)
                           & (df["basis"] == basis) & (df["ordering"] == "model")]
                    if q.empty:
                        continue
                    g = (q.groupby("frac").agg(S=("masked_frac", "mean"),
                                               F=("fid", "mean"),
                                               E=("fid", "sem"))
                           .reset_index().sort_values("S"))
                    ax.errorbar(g["S"], g["F"], yerr=g["E"], color=col, linestyle=ls,
                                marker=mk, markersize=5, linewidth=1.8, capsize=3,
                                label=lab)
                    drew = True
                if not drew:
                    plt.close(fig); print(f"  (no rows for {mode} - plot skipped)")
                    continue
                npair = df[df["mode"] == mode].groupby(["sample", "node"]).ngroups
                ax.set_xlabel("Sparsity", fontsize=11)
                ax.set_ylabel("Fidelity", fontsize=11)
                ax.set_title(f"Fidelity vs Sparsity ({mode.capitalize()})",
                             fontsize=12)
                ax.grid(alpha=.3); ax.set_axisbelow(True)
                ax.legend(fontsize=9, loc="lower right", framealpha=.92)
                fig.tight_layout()
                png = f"{OUT_DIR}/node_fidelity_{mode}_{ARM_TAG}.png"
                fig.savefig(png, dpi=300, bbox_inches="tight"); plt.close(fig)
                print(f"  plot -> {png}   ({npair:,} pairs, units = {unit})")
        except Exception as e:
            print(f"  (plotting skipped: {e})")

        print("\n  Sparsity on these axes is the FRACTION MASKED, the complement of"
              "\n  eq.26. Define it that way in the paper and do not cite eq.26 for it.")


### 16.2 Fidelity–Sparsity Curves


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE = "/speed-scratch/ar_ul/ShapleyGNN/results/FIXED_v6_no4683"
ARM  = "A_guided"
CSV  = f"{BASE}/node_fidelity/node_fidelity_owen_{ARM}.csv"
PNG  = f"{BASE}/node_fidelity/node_fidelity_eq26_{ARM}.png"

STYLE = {"owen":      dict(color="#1f4e79", ls="-",  marker="o",
                           label=r"Fid$^+$  $\Omega$"),
         "deviation": dict(color="#d95f02", ls="--", marker="s",
                           label=r"Fid$^+$  deviation")}

if not os.path.isfile(CSV):
    raise FileNotFoundError(f"{CSV} not found - run cell 115 first")
df = pd.read_csv(CSV)
HAS_EXACT = (df.estimator == "exact").any()


def estimator(mode, basis):
    if basis == "deviation":
        return "sampled"
    return "exact" if (mode == "temporal" and HAS_EXACT) else "sampled"


def curve(mode, basis):
    q = df[(df["mode"] == mode) & (df.basis == basis)
           & (df.estimator == estimator(mode, basis))
           & (df.ordering == "model")]
    return (q.groupby("frac")
             .agg(x=("sparsity", "mean"), F=("fid", "mean"), E=("fid", "sem"))
             .sort_values("x", ascending=False))


fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.1), sharey=True)
for ax, mode in zip(axes, ("spatial", "temporal")):
    for basis, st in STYLE.items():
        g = curve(mode, basis)
        ax.errorbar(g.x, g.F, yerr=g.E, markersize=5, linewidth=1.8,
                    capsize=3, **st)
    ax.set_title(f"Fidelity vs Sparsity ({mode.capitalize()})", fontsize=10.5)
    ax.set_xlabel("Sparsity", fontsize=10)
    ax.grid(alpha=.3)
    ax.set_axisbelow(True)
    ax.invert_xaxis()
axes[0].set_ylabel("Fidelity", fontsize=11)
axes[0].legend(fontsize=8.6, loc="upper left", framealpha=.92)
fig.tight_layout()
fig.savefig(PNG, dpi=300, bbox_inches="tight")
plt.close(fig)

for mode in ("spatial", "temporal"):
    o, d = curve(mode, "owen"), curve(mode, "deviation")
    n = df[df["mode"] == mode].groupby(["sample", "node"]).ngroups
    print(f"\n  {mode.upper()}   n = {n:,} pairs   "
          f"Omega estimator = {estimator(mode, 'owen')}")
    print(f"      {'sparsity':>9}{'Fid+ Om':>10}{'Fid+ dev':>10}{'dev/Om':>9}")
    for k in o.index:
        fo, fd = o.F[k], d.F[k]
        ratio = f"{fd / fo:.3f}" if fo else "-"      
        print(f"      {o.x[k]:>9.3f}{fo:>10.4f}{fd:>10.4f}{ratio:>9}")

print(f"\n  figure -> {PNG}")
